# Production Shortage Allocation & Revenue-at-Risk Optimizer

**Enterprise decision-support system for allocating constrained manufacturing supply across customer orders while protecting revenue, contribution margin, strategic customer commitments, and service performance.**

---

## Executive Summary

### Business Situation

The organization is entering the upcoming planning period with approximately **50,000 units of confirmed customer demand** against only **38,000 units of available inventory and planned production**, creating an estimated **12,000-unit supply gap** and a **24% demand shortfall**.

The constraint may originate from component shortages, supplier disruption, manufacturing capacity loss, quality holds, labor constraints, or unexpected demand acceleration. Regardless of the underlying cause, available supply is insufficient to fulfill all confirmed customer requirements.

### Management Decision

The Supply Chain and S&OP leadership team must determine how constrained supply should be allocated across **customers, orders, and product lines** before the shortage affects customer fulfillment.

The allocation decision must balance competing business priorities:

* Protect contractual customer commitments
* Preserve strategic-account service
* Prioritize economically attractive demand
* Minimize revenue and contribution margin exposure
* Reduce avoidable contract penalties
* Maintain equitable service where commercially appropriate
* Provide Sales and Customer Operations with clear fulfillment priorities

A purely first-come-first-served or uniform fair-share allocation may distribute supply without considering the economic and strategic importance of individual orders.

### Decision-Support Approach

The **Production Shortage Allocation & Revenue-at-Risk Optimizer** evaluates confirmed customer demand against available inventory and production supply and produces an order-level allocation recommendation for the constrained planning period.

The decision engine incorporates:

**Customer Priority** — strategic-account status and commercial importance

**Order Commitment** — contractual obligations and requested delivery requirements

**Product Economics** — contribution margin and revenue exposure

**Service Risk** — potential backorders, late fulfillment, and customer-service impact

**Supply Availability** — inventory, planned production, and eligible substitution opportunities

**Business Constraints** — minimum-service requirements and protected customer commitments

Multiple allocation policies are evaluated, including **FIFO, proportional fair-share, margin-focused optimization, strategic-priority allocation, and a balanced management policy**.

### Management KPIs

Allocation alternatives are evaluated using a common executive scorecard:

| Business Dimension     | KPI                              |
| ---------------------- | -------------------------------- |
| Financial Protection   | Contribution Margin Protected    |
| Revenue Exposure       | Revenue-at-Risk                  |
| Profitability Exposure | Margin-at-Risk                   |
| Customer Service       | Fill Rate                        |
| Delivery Performance   | OTIF Proxy                       |
| Unfulfilled Demand     | Backorder Rate                   |
| Contractual Risk       | Contract Penalty Exposure        |
| Strategic Accounts     | Strategic-Customer Service Level |
| Order Execution        | Perfect Order Rate               |

### Scenario & Stress Testing

The recommended allocation plan is evaluated under multiple operating conditions, including:

* Base supply-constrained plan
* Additional supply deterioration
* Unexpected demand increase
* Higher strategic-customer protection requirements
* Product-substitution recovery opportunities
* Shortage-severity sensitivity

This allows management to understand not only the recommended allocation under current assumptions, but also how the decision should change as supply and demand conditions deteriorate or recover.

### Executive Deliverables

The decision-support system provides management with:

1. **Order-Level Allocation Plan** — recommended fulfilled, partially fulfilled, and deferred quantities by customer order.
2. **Customer & Product Exposure** — identification of customers, SKUs, and commitments most affected by the shortage.
3. **Revenue & Margin Exposure** — quantified financial value associated with unfulfilled demand.
4. **Policy Comparison** — financial and service comparison across alternative allocation strategies.
5. **Shortage Stress Test** — expected performance under deeper supply constraints and demand changes.
6. **Management Action Plan** — prioritized actions for Supply Planning, Sales, Customer Operations, and Finance.
7. **Executive Control-Tower View** — consolidated visibility into supply availability, demand exposure, allocation performance, and financial consequences.

### Decision Objective

> **Allocate constrained manufacturing supply to the customer demand that best protects contractual commitments, strategic relationships, customer service, and enterprise contribution margin while making the financial consequences of every allocation decision transparent.**

---

**Data Governance Note:** This implementation uses a simulated enterprise order-management environment designed to represent realistic manufacturing planning relationships. Customer identities, orders, operational records, and financial values are synthetic; therefore, all reported business and financial impacts represent modeled outcomes rather than actual company results.


In [ ]:
# Install ../requirements.txt in your virtual environment before starting Jupyter.
import pulp
print("PuLP version:", pulp.__version__)
print("Available solvers:", pulp.listSolvers(onlyAvailable=True))


### Portable solver setup
Use Python 3.10+ and `python -m pip install -r requirements.txt` from the project root. PuLP 3.3.0 is pinned because this notebook uses its bundled `PULP_CBC_CMD` interface. No `apt-get`, privileged package installs, or in-notebook dependency upgrades are required. Run the solver smoke test below before proceeding.

In [ ]:
import pulp

model = pulp.LpProblem("Solver_Test", pulp.LpMaximize)

x = pulp.LpVariable("x", lowBound=0, cat="Integer")

model += x
model += x <= 10

solver = pulp.PULP_CBC_CMD(msg=False)
model.solve(solver)

print("Solver status:", pulp.LpStatus[model.status])
print("Test value:", x.value())

In [ ]:
# =============================================================================
# CORE ANALYTICS LIBRARIES | Supply Allocation Decision Engine
# =============================================================================
# Purpose:
# Load the data-processing, optimization, scenario-analysis, and visualization
# components used throughout the constrained-supply allocation workflow.
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

# Core data processing
import numpy as np
import pandas as pd

# Business-model structures and date handling
from datetime import datetime, timedelta

# Optimization engine
import pulp

# Executive visualization
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots


# -----------------------------------------------------------------------------
# NOTEBOOK DISPLAY CONFIGURATION
# -----------------------------------------------------------------------------

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


# -----------------------------------------------------------------------------
# ENVIRONMENT VALIDATION
# -----------------------------------------------------------------------------

print("✓ Core analytics environment initialized successfully.")
print(f"  Pandas : {pd.__version__}")
print(f"  NumPy  : {np.__version__}")
print(f"  PuLP   : {pulp.__version__}")

## Planning Assumptions & Business Rules

The decision engine centralizes all planning assumptions, allocation rules, service thresholds, and scenario parameters within a single configuration layer.

This enables Supply Chain and S&OP users to adjust management-relevant assumptions consistently across the analysis without modifying the underlying allocation logic.

Key configurable parameters include:

* Available supply and shortage severity
* Strategic-customer service requirements
* Contractual minimum-service commitments
* Customer and order priority weights
* Revenue and contribution-margin protection rules
* Backorder and contract-penalty assumptions
* Product-substitution availability
* Scenario and sensitivity settings

Centralizing these inputs ensures that all allocation policies, financial calculations, and scenario results are evaluated using a consistent set of business assumptions.


In [ ]:
# =============================================================================
# PLANNING ASSUMPTIONS & BUSINESS RULES
# Production Shortage Allocation & Revenue-at-Risk Optimizer
# =============================================================================

CONFIG = {

    # -------------------------------------------------------------------------
    # MODEL CONTROL
    # -------------------------------------------------------------------------
    "random_seed": 42,

    # -------------------------------------------------------------------------
    # PLANNING HORIZON & BUSINESS SCALE
    # -------------------------------------------------------------------------
    "n_customers": 60,
    "n_skus": 18,
    "n_orders": 220,
    "period_label": "Next 4-Week Constrained Supply Period",
    "period_weeks": 4,

    # -------------------------------------------------------------------------
    # BASE SUPPLY POSITION
    # -------------------------------------------------------------------------
    # 76% supply coverage represents an enterprise planning case in which
    # confirmed demand exceeds available inventory + planned production by 24%.
    "base_supply_factor": 0.76,

    # -------------------------------------------------------------------------
    # ORDER ECONOMICS
    # -------------------------------------------------------------------------
    "unit_price_range": (18, 420),
    "margin_pct_range": (0.12, 0.55),

    # -------------------------------------------------------------------------
    # CUSTOMER PORTFOLIO
    # -------------------------------------------------------------------------
    "strategic_customer_share": 0.20,
    "contract_customer_share": 0.45,

    # -------------------------------------------------------------------------
    # ALLOCATION POLICY & SERVICE COMMITMENTS
    # -------------------------------------------------------------------------
    "min_fill_rate_strategic": 0.90,
    "min_fill_rate_contract": 0.70,

    # Balanced allocation policy:
    # combines economic value protection with equitable supply distribution.
    "balanced_margin_weight": 0.65,
    "balanced_fairness_weight": 0.35,

    # -------------------------------------------------------------------------
    # SUPPLY & DEMAND STRESS-TEST PARAMETERS
    # -------------------------------------------------------------------------
    "scenario_supply_drop_pct": 0.20,
    "scenario_demand_surge_pct": 0.15,
    "scenario_strategic_floor_boost": 0.98,

    # -------------------------------------------------------------------------
    # SUPPLY UNCERTAINTY
    # -------------------------------------------------------------------------
    "monte_carlo_iterations": 500,
    "monte_carlo_supply_std_pct": 0.06,

    # -------------------------------------------------------------------------
    # FINANCIAL REPORTING
    # -------------------------------------------------------------------------
    "currency": "$",
}


# =============================================================================
# CONFIGURATION VALIDATION
# =============================================================================

assert 0 < CONFIG["base_supply_factor"] <= 1
assert 0 <= CONFIG["strategic_customer_share"] <= 1
assert 0 <= CONFIG["contract_customer_share"] <= 1
assert 0 <= CONFIG["min_fill_rate_strategic"] <= 1
assert 0 <= CONFIG["min_fill_rate_contract"] <= 1

assert np.isclose(
    CONFIG["balanced_margin_weight"] +
    CONFIG["balanced_fairness_weight"],
    1.0
), "Balanced-policy weights must sum to 1.0."


# Initialize reproducible simulated operating environment
np.random.seed(CONFIG["random_seed"])

base_shortage_pct = (1 - CONFIG["base_supply_factor"]) * 100

print("✓ Planning assumptions and business rules validated.")
print(f"  Planning Horizon       : {CONFIG['period_label']}")
print(f"  Customer Portfolio     : {CONFIG['n_customers']} customers")
print(f"  Product Portfolio      : {CONFIG['n_skus']} SKUs")
print(f"  Confirmed Order Base   : {CONFIG['n_orders']} orders")
print(f"  Base Supply Coverage   : {CONFIG['base_supply_factor']:.0%}")
print(f"  Base Supply Shortfall  : {base_shortage_pct:.0f}%")
print(f"  Strategic Service Floor: {CONFIG['min_fill_rate_strategic']:.0%}")
print(f"  Contract Service Floor : {CONFIG['min_fill_rate_contract']:.0%}")

## Constrained Supply Allocation Decision

### Management Question

> **When confirmed demand exceeds available supply, which customer orders should receive inventory and production capacity to best protect revenue, contribution margin, strategic customers, and contractual commitments?**

### Allocation Objective

The decision engine determines the quantity to allocate to each open customer order while balancing:

* Contribution margin protection
* Strategic-customer priority
* Contractual service commitments
* Order urgency
* Available SKU-level supply
* Customer-service impact

### Allocation Policies

Five policies are evaluated:

**FIFO** — fulfills orders based on order sequence.

**Proportional Fair Share** — distributes available supply proportionally across demand.

**Margin Protection** — prioritizes orders with greater contribution-margin value.

**Strategic Priority** — protects strategic and contractual customers.

**Balanced Policy** — balances financial value with customer-service fairness.

### Management KPIs

**Financial:** Revenue-at-Risk, Margin-at-Risk, Contribution Margin Protected, Contract Penalty Exposure

**Service:** Fill Rate, Backorder Rate, OTIF Proxy, Perfect Order Rate

**Strategic:** Strategic-Customer Service Level and Contract-Customer Service Level

### Decision Output

The system produces an **order-level allocation plan** showing recommended quantity, fill rate, backorder exposure, financial impact, and customer priority—together with a comparison of each allocation policy against FIFO.


## Enterprise Data Model

The decision engine uses a simulated ERP-style data environment representing the core relationships between **customers, sales orders, products, inventory, production supply, and approved product substitutions**.

```text
Customer ──< Sales Order >── SKU
                            ├── Inventory
                            ├── Production Plan
                            └── Approved Substitutions
```

### Core Data Tables

| Table             | Business Purpose                                                       |
| ----------------- | ---------------------------------------------------------------------- |
| `customers`       | Customer segmentation, strategic status, and contractual commitments   |
| `customer_orders` | Confirmed demand, requested quantities, due dates, and order economics |
| `inventory`       | Available SKU-level inventory                                          |
| `production_plan` | Constrained production supply available during the planning period     |
| `substitutions`   | Approved SKU substitution and supply-recovery opportunities            |

Together, these tables create the operational data foundation for **shortage identification, allocation optimization, financial exposure analysis, and management recommendations**.

> **Data Note:** All operational and financial records are synthetically generated to represent a realistic manufacturing planning environment.


In [ ]:
# =============================================================================
# CUSTOMER MASTER | Commercial & Service Segmentation
# =============================================================================

rng = np.random.default_rng(CONFIG["random_seed"])

n_cust = CONFIG["n_customers"]

# -----------------------------------------------------------------------------
# CUSTOMER SEGMENTATION
# -----------------------------------------------------------------------------

segments = rng.choice(
    ["Enterprise", "Mid-Market", "Distributor", "Small Business"],
    size=n_cust,
    p=[0.20, 0.30, 0.25, 0.25]
)

# Strategic-account likelihood varies by commercial segment
strategic_probability = {
    "Enterprise": 0.55,
    "Distributor": 0.30,
    "Mid-Market": 0.10,
    "Small Business": 0.03
}

strategic_flag = np.array([
    rng.random() < strategic_probability[segment]
    for segment in segments
])

# Align strategic-account population with planning assumptions
target_strategic = int(
    round(CONFIG["strategic_customer_share"] * n_cust)
)

current_strategic = strategic_flag.sum()

if current_strategic > target_strategic:
    eligible = np.where(strategic_flag)[0]
    remove_idx = rng.choice(
        eligible,
        size=current_strategic - target_strategic,
        replace=False
    )
    strategic_flag[remove_idx] = False

elif current_strategic < target_strategic:
    eligible = np.where(~strategic_flag)[0]
    add_idx = rng.choice(
        eligible,
        size=target_strategic - current_strategic,
        replace=False
    )
    strategic_flag[add_idx] = True


# -----------------------------------------------------------------------------
# COMMERCIAL CONTRACT PROFILE
# -----------------------------------------------------------------------------

has_contract = (
    rng.random(n_cust) <
    CONFIG["contract_customer_share"]
)

# Strategic accounts are significantly more likely to carry formal service commitments
strategic_contract_override = (
    strategic_flag &
    (rng.random(n_cust) < 0.90)
)

has_contract = has_contract | strategic_contract_override


# -----------------------------------------------------------------------------
# CUSTOMER ECONOMIC PROFILE
# -----------------------------------------------------------------------------

annual_revenue = np.where(
    strategic_flag,
    rng.lognormal(mean=15.0, sigma=0.50, size=n_cust),
    rng.lognormal(mean=12.8, sigma=0.60, size=n_cust)
).round(-3)

contract_penalty_pct = np.where(
    has_contract,
    rng.uniform(0.03, 0.12, n_cust),
    0.0
).round(3)


# -----------------------------------------------------------------------------
# CUSTOMER SERVICE TIER
# -----------------------------------------------------------------------------

customer_tier = np.select(
    [
        strategic_flag & has_contract,
        strategic_flag,
        has_contract
    ],
    [
        "Tier 1 - Strategic Contract",
        "Tier 1 - Strategic",
        "Tier 2 - Contract"
    ],
    default="Tier 3 - Standard"
)


# -----------------------------------------------------------------------------
# CUSTOMER MASTER TABLE
# -----------------------------------------------------------------------------

company_prefixes = [
    "Apex", "Northstar", "Vertex", "Summit", "Pioneer",
    "Sterling", "Atlas", "Meridian", "Crestline", "Evergreen"
]

company_suffixes = [
    "Industries", "Manufacturing", "Systems", "Group",
    "Supply Co.", "Holdings", "Technologies", "Partners"
]

customers = pd.DataFrame({
    "customer_id": [
        f"CUST-{i+1:03d}"
        for i in range(n_cust)
    ],

    "customer_name": [
        f"{rng.choice(company_prefixes)} {rng.choice(company_suffixes)}"
        for _ in range(n_cust)
    ],

    "customer_segment": segments,
    "customer_tier": customer_tier,
    "strategic_flag": strategic_flag,
    "has_contract": has_contract,
    "annual_revenue": annual_revenue,
    "contract_penalty_pct": contract_penalty_pct,
})


# -----------------------------------------------------------------------------
# CUSTOMER MASTER VALIDATION
# -----------------------------------------------------------------------------

assert customers["customer_id"].is_unique
assert customers["annual_revenue"].ge(0).all()
assert customers["contract_penalty_pct"].between(0, 1).all()

print("✓ Customer master created successfully.")
print(f"  Customers            : {len(customers):,}")
print(f"  Strategic Accounts   : {customers['strategic_flag'].sum():,}")
print(f"  Contract Customers   : {customers['has_contract'].sum():,}")
print(
    f"  Annual Revenue Base  : "
    f"${customers['annual_revenue'].sum()/1_000_000:,.1f}M"
)

customers.head(8)

In [ ]:
# =============================================================================
# PRODUCT MASTER, SUPPLY POSITION & APPROVED SUBSTITUTIONS
# =============================================================================

n_skus = CONFIG["n_skus"]
sku_ids = [f"SKU-{i+1:03d}" for i in range(n_skus)]

# -----------------------------------------------------------------------------
# PRODUCT MASTER
# -----------------------------------------------------------------------------

product_families = rng.choice(
    [
        "Controller",
        "Sensor Module",
        "Power Unit",
        "Display Assembly",
        "Connector Kit",
        "Actuator"
    ],
    size=n_skus
)

criticality = rng.choice(
    ["High", "Medium", "Low"],
    size=n_skus,
    p=[0.30, 0.45, 0.25]
)

unit_price = rng.uniform(
    *CONFIG["unit_price_range"],
    size=n_skus
).round(2)

margin_pct = rng.uniform(
    *CONFIG["margin_pct_range"],
    size=n_skus
).round(3)


# -----------------------------------------------------------------------------
# PLANNING DEMAND PROFILE
# -----------------------------------------------------------------------------

PLANNING_DEMAND_TARGET = 50_000

sku_demand_weight = rng.dirichlet(
    np.ones(n_skus) * 1.3
)

sku_planned_demand = np.maximum(
    50,
    sku_demand_weight * PLANNING_DEMAND_TARGET
).round().astype(int)


# -----------------------------------------------------------------------------
# PRODUCTION SUPPLY POSITION
# -----------------------------------------------------------------------------

# Baseline production reflects the unconstrained operating plan.
baseline_production_plan = np.maximum(
    0,
    sku_planned_demand * rng.uniform(0.95, 1.05, n_skus)
).round().astype(int)

# Available production reflects the constrained supply environment.
supply_noise = rng.normal(
    loc=1.0,
    scale=0.12,
    size=n_skus
)

available_production = np.maximum(
    0,
    sku_planned_demand
    * CONFIG["base_supply_factor"]
    * supply_noise
).round().astype(int)


# -----------------------------------------------------------------------------
# AVAILABLE INVENTORY
# -----------------------------------------------------------------------------

available_inventory = np.maximum(
    0,
    rng.poisson(
        lam=sku_planned_demand * 0.04
    )
).astype(int)


# -----------------------------------------------------------------------------
# PRODUCTION PLAN
# -----------------------------------------------------------------------------

production_plan = pd.DataFrame({
    "sku_id": sku_ids,
    "product_family": product_families,
    "criticality": criticality,
    "unit_price": unit_price,
    "contribution_margin_pct": margin_pct,
    "planned_demand": sku_planned_demand,
    "baseline_production_plan": baseline_production_plan,
    "available_production": available_production,
})


# -----------------------------------------------------------------------------
# INVENTORY POSITION
# -----------------------------------------------------------------------------

inventory = pd.DataFrame({
    "sku_id": sku_ids,
    "available_inventory": available_inventory,
})


# -----------------------------------------------------------------------------
# TOTAL AVAILABLE SUPPLY
# -----------------------------------------------------------------------------

supply_by_sku = production_plan.merge(
    inventory,
    on="sku_id",
    how="left"
)

supply_by_sku["total_available_supply"] = (
    supply_by_sku["available_production"]
    + supply_by_sku["available_inventory"]
)

supply_by_sku["supply_coverage_pct"] = (
    supply_by_sku["total_available_supply"]
    / supply_by_sku["planned_demand"]
).clip(upper=1.5)


# -----------------------------------------------------------------------------
# APPROVED PRODUCT SUBSTITUTIONS
# -----------------------------------------------------------------------------

n_substitutions = max(
    1,
    int(n_skus * 0.35)
)

substitution_sources = rng.choice(
    sku_ids,
    size=n_substitutions,
    replace=False
)

substitution_rows = []

for source_sku in substitution_sources:

    eligible_substitutes = [
        sku for sku in sku_ids
        if sku != source_sku
    ]

    substitute_sku = rng.choice(
        eligible_substitutes
    )

    conversion_ratio = round(
        float(rng.uniform(0.85, 1.00)),
        2
    )

    substitution_rows.append({
        "source_sku": source_sku,
        "substitute_sku": substitute_sku,
        "conversion_ratio": conversion_ratio
    })

substitutions = pd.DataFrame(
    substitution_rows
)


# -----------------------------------------------------------------------------
# DATA VALIDATION
# -----------------------------------------------------------------------------

assert production_plan["sku_id"].is_unique
assert inventory["sku_id"].is_unique
assert supply_by_sku["total_available_supply"].ge(0).all()

print("✓ Product and supply-position data created successfully.")
print(f"  SKUs                    : {n_skus}")
print(f"  Planning Demand         : {supply_by_sku['planned_demand'].sum():,} units")
print(f"  Available Production    : {supply_by_sku['available_production'].sum():,} units")
print(f"  Available Inventory     : {supply_by_sku['available_inventory'].sum():,} units")
print(f"  Total Available Supply  : {supply_by_sku['total_available_supply'].sum():,} units")
print(f"  Approved Substitutions  : {len(substitutions):,}")

display(
    supply_by_sku[
        [
            "sku_id",
            "product_family",
            "criticality",
            "planned_demand",
            "baseline_production_plan",
            "available_production",
            "available_inventory",
            "total_available_supply",
            "supply_coverage_pct"
        ]
    ].head(10)
)

In [ ]:
# =============================================================================
# CUSTOMER ORDER BOOK | Confirmed Demand for Constrained Planning Period
# =============================================================================

n_orders = CONFIG["n_orders"]

# -----------------------------------------------------------------------------
# ORDER HEADER ATTRIBUTES
# -----------------------------------------------------------------------------

order_customers = rng.choice(
    customers["customer_id"],
    size=n_orders
)

# SKU mix follows the planned demand profile
order_skus = rng.choice(
    sku_ids,
    size=n_orders,
    p=sku_demand_weight
)

order_qty = np.maximum(
    5,
    rng.gamma(
        shape=2.2,
        scale=90,
        size=n_orders
    )
).round().astype(int)

planning_date = datetime(2026, 8, 11)

requested_offset_days = rng.integers(
    1,
    CONFIG["period_weeks"] * 7 + 1,
    size=n_orders
)

requested_date = [
    planning_date + timedelta(days=int(days))
    for days in requested_offset_days
]


# -----------------------------------------------------------------------------
# CUSTOMER ORDER TABLE
# -----------------------------------------------------------------------------

orders = pd.DataFrame({
    "order_id": [
        f"ORD-{i+1:05d}"
        for i in range(n_orders)
    ],
    "customer_id": order_customers,
    "sku_id": order_skus,
    "order_quantity": order_qty,
    "requested_date": requested_date,
})


# -----------------------------------------------------------------------------
# PRODUCT ECONOMICS
# -----------------------------------------------------------------------------

orders = orders.merge(
    production_plan[
        [
            "sku_id",
            "unit_price",
            "contribution_margin_pct",
            "criticality"
        ]
    ],
    on="sku_id",
    how="left"
)


# -----------------------------------------------------------------------------
# CUSTOMER COMMERCIAL PROFILE
# -----------------------------------------------------------------------------

orders = orders.merge(
    customers[
        [
            "customer_id",
            "customer_name",
            "customer_segment",
            "customer_tier",
            "strategic_flag",
            "has_contract",
            "contract_penalty_pct"
        ]
    ],
    on="customer_id",
    how="left"
)


# -----------------------------------------------------------------------------
# ORDER ECONOMICS
# -----------------------------------------------------------------------------

# Represents normal customer-specific pricing variation around product list price
price_adjustment = rng.normal(
    loc=1.0,
    scale=0.05,
    size=n_orders
)

orders["selling_price"] = (
    orders["unit_price"] * price_adjustment
).clip(lower=1).round(2)

orders["order_value"] = (
    orders["selling_price"]
    * orders["order_quantity"]
).round(2)

orders["contribution_margin_per_unit"] = (
    orders["selling_price"]
    * orders["contribution_margin_pct"]
).round(2)

orders["contribution_margin_total"] = (
    orders["contribution_margin_per_unit"]
    * orders["order_quantity"]
).round(2)


# -----------------------------------------------------------------------------
# CUSTOMER / ORDER PRIORITY
# -----------------------------------------------------------------------------

orders["priority_tier"] = np.select(
    [
        orders["strategic_flag"] & orders["has_contract"],
        orders["strategic_flag"] | orders["has_contract"]
    ],
    [
        1,
        2
    ],
    default=3
)

orders["priority_label"] = orders["priority_tier"].map({
    1: "Tier 1 - Protected",
    2: "Tier 2 - Priority",
    3: "Tier 3 - Standard"
})


# -----------------------------------------------------------------------------
# DELIVERY URGENCY
# -----------------------------------------------------------------------------

orders["days_to_request"] = (
    pd.to_datetime(orders["requested_date"])
    - pd.Timestamp(planning_date)
).dt.days

orders["delivery_urgency"] = pd.cut(
    orders["days_to_request"],
    bins=[0, 7, 14, 21, np.inf],
    labels=[
        "Immediate",
        "Near-Term",
        "Mid-Horizon",
        "Later-Horizon"
    ],
    include_lowest=True
)


# -----------------------------------------------------------------------------
# ORDER EXECUTION STATUS
# -----------------------------------------------------------------------------

orders["order_status"] = "Confirmed"

# Sequence retained for FIFO benchmark policy
orders["order_sequence"] = np.arange(
    1,
    n_orders + 1
)


# -----------------------------------------------------------------------------
# FINAL CUSTOMER ORDER BOOK
# -----------------------------------------------------------------------------

orders = orders.drop(
    columns=["unit_price"]
)

assert orders["order_id"].is_unique
assert orders["order_quantity"].gt(0).all()
assert orders["order_value"].ge(0).all()

print("✓ Customer order book created successfully.")
print(f"  Confirmed Orders       : {len(orders):,}")
print(f"  Confirmed Demand       : {orders['order_quantity'].sum():,} units")
print(f"  Customer Order Value   : ${orders['order_value'].sum()/1_000_000:,.2f}M")
print(f"  Contribution Margin    : ${orders['contribution_margin_total'].sum()/1_000_000:,.2f}M")
print(
    f"  Tier-1 Protected Orders: "
    f"{(orders['priority_tier'] == 1).sum():,}"
)

display(
    orders[
        [
            "order_id",
            "customer_name",
            "sku_id",
            "order_quantity",
            "requested_date",
            "delivery_urgency",
            "priority_label",
            "order_value",
            "contribution_margin_total"
        ]
    ].head(8)
)

## Data Quality & Control Checks

Before allocation decisions are generated, the enterprise data model is validated for completeness, consistency, and operational integrity.

Key controls include:

* Missing or incomplete records
* Duplicate customer, order, and SKU identifiers
* Invalid customer-to-order or SKU relationships
* Negative or impossible quantities and financial values
* Supply values inconsistent with planning assumptions
* Extreme values requiring business review

Any issue identified is **flagged and resolved using a documented business rule** rather than being removed without explanation.


In [ ]:
# =============================================================================
# DATA QUALITY & CONTROL CHECKS
# =============================================================================

dq_issues = []

def log_dq_issue(control, count, severity, action):
    dq_issues.append({
        "control": control,
        "records_affected": int(count),
        "severity": severity,
        "action": action
    })


# -----------------------------------------------------------------------------
# 1. COMPLETENESS CHECKS
# -----------------------------------------------------------------------------

data_objects = {
    "customers": customers,
    "customer_orders": orders,
    "inventory": inventory,
    "production_plan": production_plan
}

for name, df in data_objects.items():
    missing_count = int(df.isna().sum().sum())

    log_dq_issue(
        control=f"Missing values - {name}",
        count=missing_count,
        severity="High" if missing_count > 0 else "None",
        action="Review required before allocation" if missing_count > 0 else "Passed"
    )


# -----------------------------------------------------------------------------
# 2. UNIQUE IDENTIFIER CHECKS
# -----------------------------------------------------------------------------

duplicate_orders = int(
    orders["order_id"].duplicated().sum()
)

log_dq_issue(
    control="Duplicate order identifiers",
    count=duplicate_orders,
    severity="High" if duplicate_orders > 0 else "None",
    action="Remove duplicate records after source validation" if duplicate_orders > 0 else "Passed"
)


# -----------------------------------------------------------------------------
# 3. ORDER QUANTITY VALIDATION
# -----------------------------------------------------------------------------

invalid_order_qty = int(
    (orders["order_quantity"] <= 0).sum()
)

log_dq_issue(
    control="Invalid order quantities",
    count=invalid_order_qty,
    severity="High" if invalid_order_qty > 0 else "None",
    action="Exclude from allocation pending order review" if invalid_order_qty > 0 else "Passed"
)


# -----------------------------------------------------------------------------
# 4. COMMERCIAL VALUE VALIDATION
# -----------------------------------------------------------------------------

invalid_price = int(
    (orders["selling_price"] <= 0).sum()
)

invalid_margin = int(
    (orders["contribution_margin_per_unit"] < 0).sum()
)

log_dq_issue(
    control="Invalid selling prices",
    count=invalid_price,
    severity="High" if invalid_price > 0 else "None",
    action="Exclude pending commercial master-data review" if invalid_price > 0 else "Passed"
)

log_dq_issue(
    control="Negative contribution margin",
    count=invalid_margin,
    severity="Medium" if invalid_margin > 0 else "None",
    action="Retain but flag for commercial review" if invalid_margin > 0 else "Passed"
)


# -----------------------------------------------------------------------------
# 5. REFERENTIAL INTEGRITY
# -----------------------------------------------------------------------------

unknown_customers = int(
    (~orders["customer_id"].isin(customers["customer_id"])).sum()
)

unknown_skus = int(
    (~orders["sku_id"].isin(production_plan["sku_id"])).sum()
)

log_dq_issue(
    control="Orders referencing unknown customers",
    count=unknown_customers,
    severity="High" if unknown_customers > 0 else "None",
    action="Exclude pending customer-master reconciliation" if unknown_customers > 0 else "Passed"
)

log_dq_issue(
    control="Orders referencing unknown SKUs",
    count=unknown_skus,
    severity="High" if unknown_skus > 0 else "None",
    action="Exclude pending product-master reconciliation" if unknown_skus > 0 else "Passed"
)


# -----------------------------------------------------------------------------
# 6. REQUESTED-DATE VALIDATION
# -----------------------------------------------------------------------------

planning_timestamp = pd.Timestamp(planning_date)

invalid_dates = int(
    (pd.to_datetime(orders["requested_date"]) < planning_timestamp).sum()
)

log_dq_issue(
    control="Requested dates before planning date",
    count=invalid_dates,
    severity="Medium" if invalid_dates > 0 else "None",
    action="Flag for order-management review" if invalid_dates > 0 else "Passed"
)


# -----------------------------------------------------------------------------
# 7. LARGE-ORDER EXCEPTION CHECK
# -----------------------------------------------------------------------------

qty_review_threshold = orders["order_quantity"].quantile(0.995)

large_order_flag = (
    orders["order_quantity"] > qty_review_threshold
)

large_order_count = int(
    large_order_flag.sum()
)

orders["large_order_review_flag"] = large_order_flag

log_dq_issue(
    control="Large-order exception",
    count=large_order_count,
    severity="Medium" if large_order_count > 0 else "None",
    action="Retained in demand; flagged for S&OP review" if large_order_count > 0 else "Passed"
)


# -----------------------------------------------------------------------------
# 8. SUPPLY POSITION VALIDATION
# -----------------------------------------------------------------------------

negative_production = int(
    (production_plan["available_production"] < 0).sum()
)

negative_inventory = int(
    (inventory["available_inventory"] < 0).sum()
)

log_dq_issue(
    control="Negative available production",
    count=negative_production,
    severity="High" if negative_production > 0 else "None",
    action="Correct to zero after planning-source review" if negative_production > 0 else "Passed"
)

log_dq_issue(
    control="Negative available inventory",
    count=negative_inventory,
    severity="High" if negative_inventory > 0 else "None",
    action="Correct to zero after inventory reconciliation" if negative_inventory > 0 else "Passed"
)


# -----------------------------------------------------------------------------
# DATA QUALITY SUMMARY
# -----------------------------------------------------------------------------

dq_summary = pd.DataFrame(dq_issues)

dq_summary["status"] = np.where(
    dq_summary["records_affected"] == 0,
    "PASS",
    "REVIEW"
)

print("✓ Data quality controls completed.")
print(
    f"  Controls Executed : {len(dq_summary)}"
)
print(
    f"  Controls Passed   : {(dq_summary['status'] == 'PASS').sum()}"
)
print(
    f"  Controls Requiring Review : {(dq_summary['status'] == 'REVIEW').sum()}"
)

display(dq_summary)

## Supply & Demand Exposure Assessment

Before allocation policies are evaluated, the current constrained-supply position is assessed to determine:

* Total confirmed demand versus available supply
* Overall supply shortfall and coverage
* SKUs driving the largest shortages
* Revenue and contribution margin exposed to constrained supply
* Strategic and contractual customer demand at risk
* Customers and orders with the greatest potential service exposure

This establishes the **baseline business exposure** that the allocation decision engine is designed to manage.


In [ ]:
# =============================================================================
# SUPPLY SHORTAGE ASSESSMENT | Demand vs. Available Supply by SKU
# =============================================================================

# -----------------------------------------------------------------------------
# CONFIRMED DEMAND BY SKU
# -----------------------------------------------------------------------------

demand_by_sku = (
    orders.groupby("sku_id")
    .agg(
        demand_units=("order_quantity", "sum"),
        demand_value=("order_value", "sum"),
        order_count=("order_id", "count")
    )
    .reset_index()
)


# -----------------------------------------------------------------------------
# SKU-LEVEL SUPPLY POSITION
# -----------------------------------------------------------------------------

sku_view = (
    supply_by_sku
    .merge(
        demand_by_sku,
        on="sku_id",
        how="left"
    )
)

sku_view[
    ["demand_units", "demand_value", "order_count"]
] = sku_view[
    ["demand_units", "demand_value", "order_count"]
].fillna(0)


# Shortage exists only where confirmed SKU demand exceeds available SKU supply
sku_view["shortage_units"] = (
    sku_view["demand_units"]
    - sku_view["total_available_supply"]
).clip(lower=0)

sku_view["excess_supply_units"] = (
    sku_view["total_available_supply"]
    - sku_view["demand_units"]
).clip(lower=0)

sku_view["shortage_pct"] = np.where(
    sku_view["demand_units"] > 0,
    sku_view["shortage_units"]
    / sku_view["demand_units"],
    0
)

sku_view["supply_coverage_pct"] = np.where(
    sku_view["demand_units"] > 0,
    sku_view["total_available_supply"]
    / sku_view["demand_units"],
    0
)

sku_view["supply_status"] = np.select(
    [
        sku_view["shortage_pct"] >= 0.25,
        sku_view["shortage_pct"] > 0
    ],
    [
        "Critical Shortage",
        "Constrained"
    ],
    default="Covered"
)


# -----------------------------------------------------------------------------
# ENTERPRISE SUPPLY POSITION
# -----------------------------------------------------------------------------

total_demand = sku_view["demand_units"].sum()
total_supply = sku_view["total_available_supply"].sum()

# Sum SKU-level shortages so excess supply on one SKU does not incorrectly
# offset shortages on another SKU.
total_shortage = sku_view["shortage_units"].sum()

overall_fill_potential = (
    (total_demand - total_shortage) / total_demand
    if total_demand > 0 else 0
)

shortage_rate = (
    total_shortage / total_demand
    if total_demand > 0 else 0
)

constrained_skus = (
    sku_view["shortage_units"] > 0
).sum()


# -----------------------------------------------------------------------------
# MANAGEMENT SUMMARY
# -----------------------------------------------------------------------------

print("SUPPLY SHORTAGE POSITION")
print("-" * 55)
print(f"Confirmed Demand       : {total_demand:>10,.0f} units")
print(f"Available Supply       : {total_supply:>10,.0f} units")
print(f"Uncovered Demand       : {total_shortage:>10,.0f} units")
print(f"Shortage Exposure      : {shortage_rate:>10.1%}")
print(f"Maximum Fill Potential : {overall_fill_potential:>10.1%}")
print(f"Constrained SKUs       : {constrained_skus:>10} of {len(sku_view)}")


# -----------------------------------------------------------------------------
# EXECUTIVE SUPPLY GAP VIEW
# -----------------------------------------------------------------------------

chart_data = sku_view.sort_values(
    ["shortage_pct", "shortage_units"],
    ascending=[False, False]
)

fig = px.bar(
    chart_data,
    x="sku_id",
    y=["demand_units", "total_available_supply"],
    barmode="group",
    title="Confirmed Demand vs. Available Supply by SKU",
    labels={
        "sku_id": "SKU",
        "value": "Units",
        "variable": "Supply Position"
    }
)

fig.update_layout(
    height=430,
    legend_title_text="",
    xaxis_title="SKU",
    yaxis_title="Units"
)

fig.show()


# -----------------------------------------------------------------------------
# HIGHEST SHORTAGE EXPOSURE
# -----------------------------------------------------------------------------

shortage_priority = (
    sku_view.loc[
        sku_view["shortage_units"] > 0,
        [
            "sku_id",
            "product_family",
            "criticality",
            "demand_units",
            "total_available_supply",
            "shortage_units",
            "shortage_pct",
            "supply_status"
        ]
    ]
    .sort_values(
        ["shortage_units", "shortage_pct"],
        ascending=False
    )
    .reset_index(drop=True)
)

display(shortage_priority.head(10))

In [ ]:
# =============================================================================
# CUSTOMER PORTFOLIO EXPOSURE | Strategic & Commercial Demand
# =============================================================================

# -----------------------------------------------------------------------------
# DEMAND AND FINANCIAL EXPOSURE BY CUSTOMER SEGMENT
# -----------------------------------------------------------------------------

segment_exposure = (
    orders.groupby(
        ["customer_segment", "strategic_flag"],
        as_index=False
    )
    .agg(
        order_count=("order_id", "count"),
        demand_units=("order_quantity", "sum"),
        order_value=("order_value", "sum"),
        contribution_margin=("contribution_margin_total", "sum")
    )
)

segment_exposure["customer_status"] = (
    segment_exposure["strategic_flag"]
    .map({
        True: "Strategic",
        False: "Non-Strategic"
    })
)


# -----------------------------------------------------------------------------
# STRATEGIC DEMAND EXPOSURE
# -----------------------------------------------------------------------------

strategic_demand = orders.loc[
    orders["strategic_flag"],
    "order_quantity"
].sum()

strategic_order_value = orders.loc[
    orders["strategic_flag"],
    "order_value"
].sum()

strategic_margin = orders.loc[
    orders["strategic_flag"],
    "contribution_margin_total"
].sum()

total_order_value = orders["order_value"].sum()

strategic_value_share = (
    strategic_order_value / total_order_value
    if total_order_value > 0 else 0
)


# -----------------------------------------------------------------------------
# MANAGEMENT SUMMARY
# -----------------------------------------------------------------------------

print("CUSTOMER PORTFOLIO EXPOSURE")
print("-" * 55)
print(f"Strategic Demand        : {strategic_demand:>12,.0f} units")
print(f"Strategic Order Value   : ${strategic_order_value/1_000_000:>10.2f}M")
print(f"Strategic Margin Value  : ${strategic_margin/1_000_000:>10.2f}M")
print(f"Strategic Revenue Share : {strategic_value_share:>12.1%}")


# -----------------------------------------------------------------------------
# EXECUTIVE CUSTOMER EXPOSURE VIEW
# -----------------------------------------------------------------------------

fig = px.bar(
    segment_exposure.sort_values(
        "order_value",
        ascending=True
    ),
    x="order_value",
    y="customer_segment",
    color="customer_status",
    orientation="h",
    barmode="group",
    title="Confirmed Order Value by Customer Segment and Strategic Status",
    labels={
        "order_value": "Confirmed Order Value ($)",
        "customer_segment": "Customer Segment",
        "customer_status": "Customer Status"
    },
    hover_data={
        "demand_units": ":,.0f",
        "order_count": ":,.0f",
        "contribution_margin": ":$,.0f",
        "order_value": ":$,.0f"
    }
)

fig.update_layout(
    height=430,
    legend_title_text="",
    xaxis_tickprefix="$",
    xaxis_tickformat="~s"
)

fig.show()


# -----------------------------------------------------------------------------
# CUSTOMER EXPOSURE SUMMARY
# -----------------------------------------------------------------------------

display(
    segment_exposure[
        [
            "customer_segment",
            "customer_status",
            "order_count",
            "demand_units",
            "order_value",
            "contribution_margin"
        ]
    ].sort_values(
        "order_value",
        ascending=False
    )
)

In [ ]:
# =============================================================================
# PRIORITY vs. ECONOMIC VALUE | Allocation Trade-Off Assessment
# =============================================================================

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Contribution Margin % by Customer Priority Tier",
        "Confirmed Orders by Priority Tier"
    )
)

# -----------------------------------------------------------------------------
# MARGIN DISTRIBUTION BY PRIORITY TIER
# -----------------------------------------------------------------------------

for tier in [1, 2, 3]:
    tier_orders = orders[
        orders["priority_tier"] == tier
    ]

    fig.add_trace(
        go.Box(
            y=tier_orders["contribution_margin_pct"],
            name=f"Tier {tier}",
            showlegend=False
        ),
        row=1,
        col=1
    )


# -----------------------------------------------------------------------------
# ORDER MIX BY PRIORITY TIER
# -----------------------------------------------------------------------------

tier_counts = (
    orders["priority_tier"]
    .value_counts()
    .sort_index()
)

fig.add_trace(
    go.Bar(
        x=[f"Tier {tier}" for tier in tier_counts.index],
        y=tier_counts.values,
        showlegend=False
    ),
    row=1,
    col=2
)


# -----------------------------------------------------------------------------
# EXECUTIVE FORMAT
# -----------------------------------------------------------------------------

fig.update_yaxes(
    tickformat=".0%",
    title_text="Contribution Margin %",
    row=1,
    col=1
)

fig.update_yaxes(
    title_text="Order Count",
    row=1,
    col=2
)

fig.update_layout(
    height=420,
    title_text="Customer Priority vs. Order Economics"
)

fig.show()


# -----------------------------------------------------------------------------
# MANAGEMENT SUMMARY
# -----------------------------------------------------------------------------

priority_summary = (
    orders.groupby(
        ["priority_tier", "priority_label"],
        as_index=False
    )
    .agg(
        order_count=("order_id", "count"),
        demand_units=("order_quantity", "sum"),
        order_value=("order_value", "sum"),
        contribution_margin=("contribution_margin_total", "sum"),
        avg_margin_pct=("contribution_margin_pct", "mean")
    )
)

priority_summary["revenue_share"] = (
    priority_summary["order_value"]
    / priority_summary["order_value"].sum()
)

display(priority_summary)

## Allocation Performance Framework

All allocation policies are evaluated using the same KPI framework to ensure a consistent comparison of **financial protection, customer service, and shortage exposure**.

| KPI                                  | Business Definition                                                 |
| ------------------------------------ | ------------------------------------------------------------------- |
| **Fill Rate**                        | Share of confirmed demand fulfilled during the constrained period   |
| **OTIF Proxy**                       | Share of orders receiving at least 98% of requested quantity        |
| **Backorder Rate**                   | Share of confirmed demand remaining unfulfilled                     |
| **Revenue-at-Risk**                  | Revenue value associated with unallocated demand                    |
| **Contribution Margin Protected**    | Contribution margin preserved through fulfilled demand              |
| **Lost Sales Exposure**              | Unfulfilled demand measured in units and revenue value              |
| **Contract Penalty Exposure**        | Modeled financial exposure from short-shipping contracted customers |
| **Strategic-Customer Service Level** | Fulfillment performance for strategic accounts                      |
| **Perfect Order Rate**               | Share of orders fulfilled at 100% of requested quantity             |

These measures are applied consistently across **FIFO, proportional, margin-focused, strategic-priority, and balanced allocation policies**.


In [ ]:
# =============================================================================
# ALLOCATION PERFORMANCE FRAMEWORK | KPI CALCULATION
# =============================================================================

def score_allocation(
    order_df: pd.DataFrame,
    allocation: pd.Series
):
    """
    Evaluate an order-level allocation plan using the standard financial,
    service, and customer-priority KPI framework.

    Parameters
    ----------
    order_df : pd.DataFrame
        Confirmed customer order book.

    allocation : pd.Series
        Allocated units indexed by order_id.

    Returns
    -------
    kpis : dict
        Executive KPI summary.

    allocation_detail : pd.DataFrame
        Order-level allocation and exposure detail.
    """

    df = order_df.copy()


    # -------------------------------------------------------------------------
    # ORDER-LEVEL ALLOCATION
    # -------------------------------------------------------------------------

    df["allocated_units"] = (
        df["order_id"]
        .map(allocation)
        .fillna(0)
        .clip(lower=0)
    )

    # Prevent allocation above confirmed customer demand
    df["allocated_units"] = np.minimum(
        df["allocated_units"],
        df["order_quantity"]
    )

    df["fill_rate"] = np.where(
        df["order_quantity"] > 0,
        df["allocated_units"] / df["order_quantity"],
        0
    )

    df["unfilled_units"] = (
        df["order_quantity"]
        - df["allocated_units"]
    )


    # -------------------------------------------------------------------------
    # FINANCIAL EXPOSURE
    # -------------------------------------------------------------------------

    df["revenue_protected"] = (
        df["allocated_units"]
        * df["selling_price"]
    )

    df["revenue_at_risk"] = (
        df["unfilled_units"]
        * df["selling_price"]
    )

    df["margin_protected"] = (
        df["allocated_units"]
        * df["contribution_margin_per_unit"]
    )

    df["margin_at_risk"] = (
        df["unfilled_units"]
        * df["contribution_margin_per_unit"]
    )


    # -------------------------------------------------------------------------
    # CUSTOMER SERVICE PERFORMANCE
    # -------------------------------------------------------------------------

    df["otif_flag"] = (
        df["fill_rate"] >= 0.98
    )

    df["perfect_order_flag"] = (
        df["fill_rate"] >= 0.999
    )

    df["backorder_flag"] = (
        df["unfilled_units"] > 0
    )


    # -------------------------------------------------------------------------
    # CONTRACTUAL SERVICE EXPOSURE
    # -------------------------------------------------------------------------

    # Contract customers are measured against the configured contractual
    # service threshold rather than requiring 100% fulfillment.
    contract_service_floor = CONFIG["min_fill_rate_contract"]

    df["contract_service_breach"] = (
        df["has_contract"]
        & (df["fill_rate"] < contract_service_floor)
    )

    df["contract_penalty_exposure"] = np.where(
        df["contract_service_breach"],
        df["revenue_at_risk"]
        * df["contract_penalty_pct"],
        0.0
    )


    # -------------------------------------------------------------------------
    # ENTERPRISE KPI SUMMARY
    # -------------------------------------------------------------------------

    total_demand_units = df["order_quantity"].sum()
    total_allocated_units = df["allocated_units"].sum()
    total_demand_value = df["order_value"].sum()

    strategic_df = df[
        df["strategic_flag"]
    ]

    contract_df = df[
        df["has_contract"]
    ]

    strategic_service_level = (
        strategic_df["allocated_units"].sum()
        / strategic_df["order_quantity"].sum()
        if strategic_df["order_quantity"].sum() > 0
        else np.nan
    )

    contract_service_level = (
        contract_df["allocated_units"].sum()
        / contract_df["order_quantity"].sum()
        if contract_df["order_quantity"].sum() > 0
        else np.nan
    )

    fill_rate = (
        total_allocated_units / total_demand_units
        if total_demand_units > 0
        else 0
    )


    kpis = {

        # Service
        "fill_rate": fill_rate,
        "otif_rate": df["otif_flag"].mean(),
        "backorder_rate": 1 - fill_rate,
        "perfect_order_rate": df["perfect_order_flag"].mean(),

        # Customer priority
        "strategic_customer_service_level": strategic_service_level,
        "contract_customer_service_level": contract_service_level,

        # Financial
        "revenue_protected": df["revenue_protected"].sum(),
        "revenue_at_risk": df["revenue_at_risk"].sum(),
        "contribution_margin_protected": df["margin_protected"].sum(),
        "margin_at_risk": df["margin_at_risk"].sum(),
        "contract_penalty_exposure": df["contract_penalty_exposure"].sum(),

        # Operational
        "lost_sales_units": df["unfilled_units"].sum(),
        "backordered_orders": df["backorder_flag"].sum(),

        # Scale / reference
        "total_demand_units": total_demand_units,
        "total_allocated_units": total_allocated_units,
        "total_demand_value": total_demand_value,
    }

    return kpis, df


# =============================================================================
# KPI COMPARISON TABLE HELPER
# =============================================================================

def kpis_to_frame(
    kpi_dict: dict,
    policy_name: str
) -> pd.DataFrame:

    return pd.DataFrame([
        {
            "policy": policy_name,
            **kpi_dict
        }
    ])


print("✓ Allocation performance framework initialized.")

## Baseline Allocation Policies

Two benchmark allocation policies establish the baseline for evaluating the financial and service impact of more advanced allocation strategies.

**FIFO — First-Come, First-Served**
Available supply is allocated according to order-entry sequence until SKU-level supply is exhausted. This policy prioritizes order timing without considering margin, customer tier, or strategic importance.

**Proportional Fair Share**
Available supply is distributed proportionally across all open orders for each constrained SKU. Every eligible order receives the same SKU-level fulfillment percentage.

These benchmarks provide a consistent reference point for measuring the incremental value of the optimized allocation policies.


In [ ]:
# =============================================================================
# FIFO BASELINE POLICY | First-Come, First-Served Allocation
# =============================================================================

def allocate_fifo(
    order_df: pd.DataFrame,
    supply_df: pd.DataFrame
) -> pd.Series:
    """
    Allocate available SKU-level supply according to order-entry sequence.

    Policy logic:
    - Earlier confirmed orders receive supply first.
    - Allocation cannot exceed order quantity.
    - Allocation cannot exceed remaining SKU-level supply.
    """

    remaining_supply = (
        supply_df
        .set_index("sku_id")["total_available_supply"]
        .astype(float)
        .to_dict()
    )

    allocation = {}

    fifo_order_book = order_df.sort_values(
        "order_sequence",
        ascending=True
    )

    for order in fifo_order_book.itertuples():

        sku_id = order.sku_id
        order_id = order.order_id
        requested_qty = float(order.order_quantity)

        available_qty = max(
            remaining_supply.get(sku_id, 0),
            0
        )

        allocated_qty = min(
            requested_qty,
            available_qty
        )

        allocation[order_id] = allocated_qty

        remaining_supply[sku_id] = (
            available_qty - allocated_qty
        )

    allocation_series = pd.Series(
        allocation,
        name="allocated_units",
        dtype=float
    )

    return allocation_series


# =============================================================================
# FIFO BASELINE EVALUATION
# =============================================================================

alloc_fifo = allocate_fifo(
    orders,
    sku_view
)

kpi_fifo, detail_fifo = score_allocation(
    orders,
    alloc_fifo
)

fifo_summary = kpis_to_frame(
    kpi_fifo,
    "FIFO"
)

display(fifo_summary)

In [ ]:
# =============================================================================
# PROPORTIONAL FAIR-SHARE BASELINE | SKU-Level Allocation
# =============================================================================

def allocate_proportional(
    order_df: pd.DataFrame,
    supply_df: pd.DataFrame
) -> pd.Series:
    """
    Allocate available SKU-level supply proportionally across all open orders.

    Policy logic:
    - All orders for the same SKU receive the same fulfillment percentage.
    - Allocation cannot exceed confirmed order quantity.
    - Allocation cannot exceed available SKU-level supply.
    """

    supply_map = (
        supply_df
        .set_index("sku_id")["total_available_supply"]
        .astype(float)
        .to_dict()
    )

    demand_map = (
        order_df
        .groupby("sku_id")["order_quantity"]
        .sum()
        .to_dict()
    )

    allocation = {}

    for order in order_df.itertuples():

        sku_id = order.sku_id
        requested_qty = float(order.order_quantity)

        sku_supply = max(
            supply_map.get(sku_id, 0),
            0
        )

        sku_demand = max(
            demand_map.get(sku_id, 0),
            0
        )

        fill_share = (
            min(sku_supply / sku_demand, 1.0)
            if sku_demand > 0
            else 0
        )

        allocated_qty = np.floor(
            requested_qty * fill_share
        )

        allocation[order.order_id] = allocated_qty

    allocation_series = pd.Series(
        allocation,
        name="allocated_units",
        dtype=float
    )

    return allocation_series


# =============================================================================
# PROPORTIONAL FAIR-SHARE EVALUATION
# =============================================================================

alloc_prop = allocate_proportional(
    orders,
    sku_view
)

kpi_prop, detail_prop = score_allocation(
    orders,
    alloc_prop
)

prop_summary = kpis_to_frame(
    kpi_prop,
    "Proportional Fair Share"
)

display(prop_summary)

## Optimization Model — Margin Protection Allocation

The optimization engine determines the order-level allocation plan that protects the greatest contribution margin while remaining within available SKU supply and approved customer-service commitments.

### Decision Variable

For each customer order (i):

[
x_i = \text{units allocated to order } i
]

with:

[
0 \le x_i \le q_i
]

where (q_i) is the confirmed order quantity.

Split fulfillment is permitted, allowing an order to be partially supplied during the constrained period.

### Primary Objective

[
\max \sum_i m_i x_i
]

where (m_i) represents the contribution margin per unit of order (i).

The objective therefore prioritizes the available supply toward demand that protects the greatest economic value.

### Business Constraints

**SKU Supply Constraint**

[
\sum_{i \in SKU_s} x_i \le S_s
]

Total allocation for each SKU cannot exceed available inventory and constrained production supply.

**Order Quantity Constraint**

[
x_i \le q_i
]

No order can receive more than its confirmed requirement.

**Strategic Customer Protection**

Strategic customers must meet the approved minimum service threshold where the available supply position remains feasible.

**Contract Customer Protection**

Contract customers receive minimum service protection to reduce avoidable contractual and financial exposure.

**Allocation Integrity**

[
x_i \ge 0
]

All allocation decisions must remain operationally valid.

### Balanced Allocation Variant

A second optimization policy evaluates a blended objective that balances:

**Contribution Margin Protection + Customer-Service Fairness**

This allows management to compare a financially aggressive allocation plan with a more balanced plan that protects both economic value and customer relationships.


In [ ]:
# =============================================================================
# ALLOCATION OPTIMIZATION ENGINE
# Margin Protection | Customer Commitments | Service Fairness
# =============================================================================

def optimize_allocation(
    order_df: pd.DataFrame,
    supply_df: pd.DataFrame,
    min_fill_strategic: float,
    min_fill_contract: float,
    objective_mode: str = "margin",
    verbose: bool = False
):
    """
    Generate an order-level constrained-supply allocation plan.

    Supported policies
    ------------------
    margin:
        Prioritizes contribution-margin protection while accounting for
        strategic and contractual service commitments.

    balanced:
        Combines contribution-margin protection with broader customer-service
        fairness.

    Returns
    -------
    allocation : pd.Series
        Recommended allocated units indexed by order_id.

    solver_status : str
        Optimization status returned by CBC.

    model_info : dict
        Model size, objective value, and service-shortfall information.
    """

    if objective_mode not in {"margin", "balanced"}:
        raise ValueError(
            "objective_mode must be either 'margin' or 'balanced'."
        )


    # -------------------------------------------------------------------------
    # MODEL INPUTS
    # -------------------------------------------------------------------------

    # Fail before optimization if missing supply would leave orders unconstrained.
    if order_df.empty or supply_df.empty:
        raise ValueError("Orders and supply must be nonempty.")
    for frame, key in [(order_df, "order_id"), (supply_df, "sku_id")]:
        if frame[key].isna().any() or frame[key].duplicated().any():
            raise ValueError(f"{key} must be non-null and unique.")
    if order_df["sku_id"].isna().any() or not set(order_df["sku_id"]).issubset(set(supply_df["sku_id"])):
        raise ValueError("Every ordered SKU must have an explicit supply row (zero is valid).")
    for frame, field in [(order_df, "order_quantity"), (supply_df, "total_available_supply")]:
        values = frame[field].to_numpy(dtype=float)
        if not np.isfinite(values).all() or (values < 0).any() or (values != np.floor(values)).any():
            raise ValueError(f"{field} must contain finite, nonnegative integer units.")
    if not np.isfinite(order_df["contribution_margin_per_unit"].to_numpy(dtype=float)).all():
        raise ValueError("Unit contribution margins must be finite.")
    for target in (min_fill_strategic, min_fill_contract):
        if not np.isfinite(target) or not 0 <= target <= 1:
            raise ValueError("Service targets must be between zero and one.")

    order_ids = order_df["order_id"].tolist()

    order_qty = (
        order_df
        .set_index("order_id")["order_quantity"]
        .to_dict()
    )

    margin_per_unit = (
        order_df
        .set_index("order_id")["contribution_margin_per_unit"]
        .to_dict()
    )

    sku_by_order = (
        order_df
        .set_index("order_id")["sku_id"]
        .to_dict()
    )

    supply_map = (
        supply_df
        .set_index("sku_id")["total_available_supply"]
        .to_dict()
    )


    # -------------------------------------------------------------------------
    # OPTIMIZATION MODEL
    # -------------------------------------------------------------------------

    model = pulp.LpProblem(
        "Production_Shortage_Allocation",
        pulp.LpMaximize
    )


    # -------------------------------------------------------------------------
    # ORDER-LEVEL ALLOCATION VARIABLES
    # -------------------------------------------------------------------------
    # Integer units are used because physical manufactured products are discrete.
    # -------------------------------------------------------------------------

    allocation_var = {
        order_id: pulp.LpVariable(
            f"allocate_{order_id}",
            lowBound=0,
            upBound=int(order_qty[order_id]),
            cat="Integer"
        )
        for order_id in order_ids
    }


    # -------------------------------------------------------------------------
    # SKU SUPPLY CONSTRAINTS
    # -------------------------------------------------------------------------

    for sku_id, available_supply in supply_map.items():

        sku_orders = [
            order_id
            for order_id in order_ids
            if sku_by_order[order_id] == sku_id
        ]

        if sku_orders:

            model += (
                pulp.lpSum(
                    allocation_var[order_id]
                    for order_id in sku_orders
                )
                <= int(np.floor(available_supply)),
                f"Supply_{sku_id}"
            )


    # -------------------------------------------------------------------------
    # CUSTOMER SERVICE COMMITMENTS
    # -------------------------------------------------------------------------
    # Service thresholds are modeled with explicit shortfall variables.
    # This prevents severe shortage scenarios from making the model infeasible
    # while making any service breach visible and economically penalized.
    # -------------------------------------------------------------------------

    customer_shortfall = {}

    customer_groups = {
        customer_id: group.copy()
        for customer_id, group
        in order_df.groupby("customer_id")
    }

    for customer_id, group in customer_groups.items():

        customer_orders = group["order_id"].tolist()
        customer_demand = float(
            group["order_quantity"].sum()
        )

        is_strategic = bool(
            group["strategic_flag"].iloc[0]
        )

        has_contract = bool(
            group["has_contract"].iloc[0]
        )

        if is_strategic:
            service_target = min_fill_strategic

        elif has_contract:
            service_target = min_fill_contract

        else:
            continue

        customer_shortfall[customer_id] = pulp.LpVariable(
            f"service_shortfall_{customer_id}",
            lowBound=0,
            cat="Continuous"
        )

        model += (
            pulp.lpSum(
                allocation_var[order_id]
                for order_id in customer_orders
            )
            + customer_shortfall[customer_id]
            >= service_target * customer_demand,
            f"ServiceCommitment_{customer_id}"
        )


    # -------------------------------------------------------------------------
    # CUSTOMER FAIRNESS VARIABLE
    # -------------------------------------------------------------------------
    # Represents the minimum fulfillment percentage achieved across customers.
    # Used only by the balanced allocation policy.
    # -------------------------------------------------------------------------

    minimum_customer_fill = pulp.LpVariable(
        "minimum_customer_fill",
        lowBound=0,
        upBound=1,
        cat="Continuous"
    )

    for customer_id, group in customer_groups.items():

        customer_orders = group["order_id"].tolist()
        customer_demand = float(
            group["order_quantity"].sum()
        )

        if customer_demand > 0:

            model += (
                pulp.lpSum(
                    allocation_var[order_id]
                    for order_id in customer_orders
                )
                >= minimum_customer_fill * customer_demand,
                f"CustomerFairness_{customer_id}"
            )


    # -------------------------------------------------------------------------
    # OBJECTIVE COMPONENTS
    # -------------------------------------------------------------------------

    protected_margin = pulp.lpSum(
        margin_per_unit[order_id]
        * allocation_var[order_id]
        for order_id in order_ids
    )

    # Service shortfalls receive a strong economic penalty.
    # Strategic-account shortfalls carry greater weight.
    service_penalty_terms = []

    for customer_id, shortfall_var in customer_shortfall.items():

        customer_group = customer_groups[customer_id]

        is_strategic = bool(
            customer_group["strategic_flag"].iloc[0]
        )

        penalty_weight = (
            2.0 if is_strategic else 1.0
        )

        avg_margin = (
            customer_group[
                "contribution_margin_per_unit"
            ].mean()
        )

        service_penalty_terms.append(
            penalty_weight
            * avg_margin
            * shortfall_var
        )

    service_penalty = (
        pulp.lpSum(service_penalty_terms)
        if service_penalty_terms
        else 0
    )


    # -------------------------------------------------------------------------
    # POLICY OBJECTIVE
    # -------------------------------------------------------------------------

    if objective_mode == "margin":

        model += (
            protected_margin
            - service_penalty
        )

    else:

        margin_weight = CONFIG[
            "balanced_margin_weight"
        ]

        fairness_weight = CONFIG[
            "balanced_fairness_weight"
        ]

        # Scale fairness into approximately the same financial range
        # as the margin objective.
        total_margin_opportunity = (
            order_df[
                "contribution_margin_total"
            ].sum()
        )

        fairness_value = (
            minimum_customer_fill
            * total_margin_opportunity
        )

        model += (
            margin_weight * protected_margin
            + fairness_weight * fairness_value
            - service_penalty
        )


    # -------------------------------------------------------------------------
    # SOLVE
    # -------------------------------------------------------------------------

    solver = pulp.PULP_CBC_CMD(
        msg=1 if verbose else 0
    )

    model.solve(solver)

    solver_status = pulp.LpStatus[
        model.status
    ]
    if solver_status != "Optimal":
        raise RuntimeError(f"Allocation solve failed: {solver_status}")


    # -------------------------------------------------------------------------
    # RESULT EXTRACTION
    # -------------------------------------------------------------------------

    allocation = pd.Series(
        {
            order_id: max(
                float(allocation_var[order_id].value() or 0),
                0
            )
            for order_id in order_ids
        },
        name="allocated_units"
    )


    service_shortfall_units = {
        customer_id: float(
            variable.value() or 0
        )
        for customer_id, variable
        in customer_shortfall.items()
    }


    model_info = {
        "objective_mode": objective_mode,
        "solver_status": solver_status,
        "n_allocation_variables": len(allocation_var),
        "n_constraints": len(model.constraints),
        "objective_value": pulp.value(model.objective),
        "minimum_customer_fill": float(
            minimum_customer_fill.value() or 0
        ),
        "service_shortfall_units": service_shortfall_units,
        "total_service_shortfall_units": sum(
            service_shortfall_units.values()
        )
    }


    return allocation, solver_status, model_info


print("✓ Constrained-supply allocation optimization engine initialized.")

In [ ]:
# =============================================================================
# MARGIN PROTECTION POLICY | Optimization Run & Validation
# =============================================================================

alloc_margin, status_margin, info_margin = optimize_allocation(
    orders,
    sku_view,
    min_fill_strategic=CONFIG["min_fill_rate_strategic"],
    min_fill_contract=CONFIG["min_fill_rate_contract"],
    objective_mode="margin"
)


# -----------------------------------------------------------------------------
# SOLVER STATUS
# -----------------------------------------------------------------------------

print("MARGIN PROTECTION POLICY")
print("-" * 60)
print(f"Solver Status            : {status_margin}")
print(f"Allocation Variables     : {info_margin['n_allocation_variables']:,}")
print(f"Model Constraints        : {info_margin['n_constraints']:,}")
print(f"Objective Value          : {info_margin['objective_value']:,.2f}")


# -----------------------------------------------------------------------------
# KPI EVALUATION
# -----------------------------------------------------------------------------

kpi_margin, detail_margin = score_allocation(
    orders,
    alloc_margin
)


# -----------------------------------------------------------------------------
# ALLOCATION INTEGRITY CHECKS
# -----------------------------------------------------------------------------

assert status_margin in {
    "Optimal",
    "Feasible"
}, f"Optimization did not return a usable solution: {status_margin}"

assert (
    kpi_margin["total_allocated_units"]
    <= kpi_margin["total_demand_units"] + 1e-6
), "Allocation exceeds confirmed demand."


# SKU-level supply validation
sku_check = (
    detail_margin
    .groupby("sku_id", as_index=False)
    .agg(
        allocated_units=("allocated_units", "sum")
    )
    .merge(
        sku_view[
            [
                "sku_id",
                "total_available_supply"
            ]
        ],
        on="sku_id",
        how="left"
    )
)

sku_check["capacity_variance"] = (
    sku_check["total_available_supply"]
    - sku_check["allocated_units"]
)

capacity_violations = sku_check[
    sku_check["allocated_units"]
    > sku_check["total_available_supply"] + 1e-6
]

assert capacity_violations.empty, (
    "SKU-level allocation exceeds available supply."
)

print(f"SKU Supply Violations    : {len(capacity_violations)}")


# -----------------------------------------------------------------------------
# CUSTOMER SERVICE COMMITMENT REVIEW
# -----------------------------------------------------------------------------

customer_service = (
    detail_margin
    .groupby(
        [
            "customer_id",
            "customer_name",
            "strategic_flag",
            "has_contract"
        ],
        as_index=False
    )
    .agg(
        demand_units=("order_quantity", "sum"),
        allocated_units=("allocated_units", "sum")
    )
)

customer_service["fill_rate"] = np.where(
    customer_service["demand_units"] > 0,
    customer_service["allocated_units"]
    / customer_service["demand_units"],
    0
)

strategic_service = customer_service[
    customer_service["strategic_flag"]
].copy()

contract_service = customer_service[
    customer_service["has_contract"]
].copy()

strategic_breaches = strategic_service[
    strategic_service["fill_rate"]
    < CONFIG["min_fill_rate_strategic"]
]

contract_breaches = contract_service[
    contract_service["fill_rate"]
    < CONFIG["min_fill_rate_contract"]
]


print(
    f"Strategic Service Level  : "
    f"{kpi_margin['strategic_customer_service_level']:.1%}"
)

print(
    f"Contract Service Level   : "
    f"{kpi_margin['contract_customer_service_level']:.1%}"
)

print(
    f"Strategic Target Breaches: "
    f"{len(strategic_breaches)}"
)

print(
    f"Contract Target Breaches : "
    f"{len(contract_breaches)}"
)

print(
    f"Service Shortfall Units  : "
    f"{info_margin['total_service_shortfall_units']:,.0f}"
)


# -----------------------------------------------------------------------------
# MANAGEMENT PERFORMANCE SUMMARY
# -----------------------------------------------------------------------------

print("\nFINANCIAL & SERVICE OUTCOME")
print("-" * 60)

print(
    f"Contribution Margin Protected : "
    f"{CONFIG['currency']}"
    f"{kpi_margin['contribution_margin_protected']:,.0f}"
)

print(
    f"Revenue at Risk              : "
    f"{CONFIG['currency']}"
    f"{kpi_margin['revenue_at_risk']:,.0f}"
)

print(
    f"Margin at Risk               : "
    f"{CONFIG['currency']}"
    f"{kpi_margin['margin_at_risk']:,.0f}"
)

print(
    f"Overall Fill Rate            : "
    f"{kpi_margin['fill_rate']:.1%}"
)

print(
    f"Backordered Units            : "
    f"{kpi_margin['lost_sales_units']:,.0f}"
)

print(
    f"Contract Penalty Exposure    : "
    f"{CONFIG['currency']}"
    f"{kpi_margin['contract_penalty_exposure']:,.0f}"
)


# -----------------------------------------------------------------------------
# POLICY SUMMARY
# -----------------------------------------------------------------------------

margin_summary = kpis_to_frame(
    kpi_margin,
    "Margin Protection"
)

display(margin_summary)





**Management Interpretation:** The Margin Protection policy directs constrained supply toward orders that preserve the greatest contribution margin while applying elevated protection to strategic and contractual customers. Any service commitments that cannot be achieved because of SKU-level shortages remain explicitly visible as management exceptions rather than making the allocation model infeasible.


**Management Interpretation:** This policy tests the financial and service impact of placing near-full protection on strategic accounts. It allows leadership to quantify the incremental cost of stronger strategic-customer protection relative to the standard Margin Protection policy.


In [ ]:
# =============================================================================
# STRATEGIC CUSTOMER PROTECTION POLICY | Optimization Run
# =============================================================================

strategic_service_target = 0.99

alloc_strategic, status_strategic, info_strategic = optimize_allocation(
    orders,
    sku_view,
    min_fill_strategic=strategic_service_target,
    min_fill_contract=CONFIG["min_fill_rate_contract"],
    objective_mode="margin"
)


# -----------------------------------------------------------------------------
# POLICY EVALUATION
# -----------------------------------------------------------------------------

kpi_strategic, detail_strategic = score_allocation(
    orders,
    alloc_strategic
)

print("STRATEGIC CUSTOMER PROTECTION POLICY")
print("-" * 60)
print(f"Solver Status             : {status_strategic}")
print(f"Strategic Service Target  : {strategic_service_target:.0%}")
print(
    f"Strategic Service Achieved: "
    f"{kpi_strategic['strategic_customer_service_level']:.1%}"
)
print(
    f"Overall Fill Rate         : "
    f"{kpi_strategic['fill_rate']:.1%}"
)
print(
    f"Contribution Margin       : "
    f"{CONFIG['currency']}"
    f"{kpi_strategic['contribution_margin_protected']:,.0f}"
)
print(
    f"Revenue at Risk           : "
    f"{CONFIG['currency']}"
    f"{kpi_strategic['revenue_at_risk']:,.0f}"
)
print(
    f"Service Shortfall Units   : "
    f"{info_strategic['total_service_shortfall_units']:,.0f}"
)


# -----------------------------------------------------------------------------
# POLICY SUMMARY
# -----------------------------------------------------------------------------

strategic_summary = kpis_to_frame(
    kpi_strategic,
    "Strategic Customer Protection"
)

display(strategic_summary)

In [ ]:
# =============================================================================
# BALANCED ALLOCATION POLICY
# Contribution Margin Protection + Customer Service Fairness
# =============================================================================

# -----------------------------------------------------------------------------
# MODEL INPUTS
# -----------------------------------------------------------------------------

order_ids = orders["order_id"].tolist()

order_qty = (
    orders
    .set_index("order_id")["order_quantity"]
    .astype(float)
    .to_dict()
)

margin_per_unit = (
    orders
    .set_index("order_id")["contribution_margin_per_unit"]
    .astype(float)
    .to_dict()
)

sku_by_order = (
    orders
    .set_index("order_id")["sku_id"]
    .to_dict()
)

supply_map = (
    sku_view
    .set_index("sku_id")["total_available_supply"]
    .astype(float)
    .to_dict()
)


# -----------------------------------------------------------------------------
# CREATE MODEL
# -----------------------------------------------------------------------------

balanced_model = pulp.LpProblem(
    "Balanced_Supply_Allocation",
    pulp.LpMaximize
)


# -----------------------------------------------------------------------------
# ORDER-LEVEL ALLOCATION VARIABLES
# Split shipments are allowed, so continuous quantities are appropriate.
# -----------------------------------------------------------------------------

x = {
    order_id: pulp.LpVariable(
        f"alloc_{order_id}",
        lowBound=0,
        upBound=order_qty[order_id],
        cat="Continuous"
    )
    for order_id in order_ids
}


# -----------------------------------------------------------------------------
# SKU SUPPLY CONSTRAINTS
# -----------------------------------------------------------------------------

for sku_id, available_supply in supply_map.items():

    sku_orders = [
        order_id
        for order_id in order_ids
        if sku_by_order[order_id] == sku_id
    ]

    if sku_orders:
        balanced_model += (
            pulp.lpSum(
                x[order_id]
                for order_id in sku_orders
            )
            <= available_supply,
            f"Supply_{sku_id}"
        )


# -----------------------------------------------------------------------------
# CUSTOMER GROUPS
# -----------------------------------------------------------------------------

customer_groups = {
    customer_id: group.copy()
    for customer_id, group
    in orders.groupby("customer_id")
}


# -----------------------------------------------------------------------------
# FAIRNESS VARIABLE
# Minimum fulfillment percentage achieved across customers.
# -----------------------------------------------------------------------------

minimum_customer_fill = pulp.LpVariable(
    "minimum_customer_fill",
    lowBound=0,
    upBound=1,
    cat="Continuous"
)

for customer_id, group in customer_groups.items():

    customer_orders = group["order_id"].tolist()
    customer_demand = float(
        group["order_quantity"].sum()
    )

    if customer_demand > 0:
        balanced_model += (
            pulp.lpSum(
                x[order_id]
                for order_id in customer_orders
            )
            >= minimum_customer_fill * customer_demand,
            f"Fairness_{customer_id}"
        )


# -----------------------------------------------------------------------------
# STRATEGIC / CONTRACT SERVICE COMMITMENTS
# Soft constraints keep the model feasible during severe shortages.
# -----------------------------------------------------------------------------

service_shortfall = {}

for customer_id, group in customer_groups.items():

    customer_orders = group["order_id"].tolist()
    customer_demand = float(
        group["order_quantity"].sum()
    )

    is_strategic = bool(
        group["strategic_flag"].iloc[0]
    )

    has_contract = bool(
        group["has_contract"].iloc[0]
    )

    if is_strategic:
        service_target = CONFIG["min_fill_rate_strategic"]

    elif has_contract:
        service_target = CONFIG["min_fill_rate_contract"]

    else:
        continue

    shortfall_var = pulp.LpVariable(
        f"service_shortfall_{customer_id}",
        lowBound=0,
        cat="Continuous"
    )

    service_shortfall[customer_id] = shortfall_var

    balanced_model += (
        pulp.lpSum(
            x[order_id]
            for order_id in customer_orders
        )
        + shortfall_var
        >= service_target * customer_demand,
        f"ServiceCommitment_{customer_id}"
    )


# -----------------------------------------------------------------------------
# OBJECTIVE COMPONENT 1: CONTRIBUTION MARGIN PROTECTION
# -----------------------------------------------------------------------------

protected_margin = pulp.lpSum(
    margin_per_unit[order_id]
    * x[order_id]
    for order_id in order_ids
)


# -----------------------------------------------------------------------------
# OBJECTIVE COMPONENT 2: CUSTOMER FAIRNESS
# Scale fairness into the same approximate economic range as the margin term.
# -----------------------------------------------------------------------------

total_margin_opportunity = float(
    orders["contribution_margin_total"].sum()
)

fairness_value = (
    minimum_customer_fill
    * total_margin_opportunity
)


# -----------------------------------------------------------------------------
# OBJECTIVE COMPONENT 3: SERVICE SHORTFALL PENALTY
# Strategic customers receive a higher penalty weight.
# -----------------------------------------------------------------------------

service_penalty_terms = []

for customer_id, shortfall_var in service_shortfall.items():

    group = customer_groups[customer_id]

    avg_margin_per_unit = float(
        group["contribution_margin_per_unit"].mean()
    )

    is_strategic = bool(
        group["strategic_flag"].iloc[0]
    )

    penalty_multiplier = (
        2.0 if is_strategic else 1.0
    )

    service_penalty_terms.append(
        penalty_multiplier
        * avg_margin_per_unit
        * shortfall_var
    )

service_penalty = (
    pulp.lpSum(service_penalty_terms)
    if service_penalty_terms
    else 0
)


# -----------------------------------------------------------------------------
# BALANCED BUSINESS OBJECTIVE
# -----------------------------------------------------------------------------

margin_weight = CONFIG["balanced_margin_weight"]
fairness_weight = CONFIG["balanced_fairness_weight"]

balanced_model += (
    margin_weight * protected_margin
    + fairness_weight * fairness_value
    - service_penalty
)


# -----------------------------------------------------------------------------
# SOLVE
# -----------------------------------------------------------------------------

solver = pulp.PULP_CBC_CMD(
    msg=False,
    timeLimit=30
)

balanced_model.solve(solver)

status_balanced = pulp.LpStatus[
    balanced_model.status
]

print("BALANCED ALLOCATION POLICY")
print("-" * 60)
print(f"Solver Status : {status_balanced}")


# -----------------------------------------------------------------------------
# VALIDATE SOLVER STATUS
# -----------------------------------------------------------------------------

if status_balanced not in ["Optimal", "Feasible"]:
    raise RuntimeError(
        f"Balanced allocation did not return a usable solution. "
        f"Solver status: {status_balanced}"
    )


# -----------------------------------------------------------------------------
# EXTRACT ORDER-LEVEL ALLOCATION
# -----------------------------------------------------------------------------

alloc_balanced = pd.Series(
    {
        order_id: max(
            float(x[order_id].value() or 0),
            0
        )
        for order_id in order_ids
    },
    name="allocated_units"
)


# -----------------------------------------------------------------------------
# SCORE BUSINESS PERFORMANCE
# -----------------------------------------------------------------------------

kpi_balanced, detail_balanced = score_allocation(
    orders,
    alloc_balanced
)


# -----------------------------------------------------------------------------
# SERVICE SHORTFALL SUMMARY
# -----------------------------------------------------------------------------

service_shortfall_units = sum(
    float(var.value() or 0)
    for var in service_shortfall.values()
)

minimum_fill_achieved = float(
    minimum_customer_fill.value() or 0
)


# -----------------------------------------------------------------------------
# MANAGEMENT OUTCOME
# -----------------------------------------------------------------------------

print("\nBALANCED ALLOCATION OUTCOME")
print("-" * 60)

print(
    f"Overall Fill Rate            : "
    f"{kpi_balanced['fill_rate']:.1%}"
)

print(
    f"Strategic Customer Service   : "
    f"{kpi_balanced['strategic_customer_service_level']:.1%}"
)

print(
    f"Contract Customer Service    : "
    f"{kpi_balanced['contract_customer_service_level']:.1%}"
)

print(
    f"Minimum Customer Fill        : "
    f"{minimum_fill_achieved:.1%}"
)

print(
    f"Contribution Margin Protected: "
    f"{CONFIG['currency']}"
    f"{kpi_balanced['contribution_margin_protected']:,.0f}"
)

print(
    f"Revenue at Risk              : "
    f"{CONFIG['currency']}"
    f"{kpi_balanced['revenue_at_risk']:,.0f}"
)

print(
    f"Margin at Risk               : "
    f"{CONFIG['currency']}"
    f"{kpi_balanced['margin_at_risk']:,.0f}"
)

print(
    f"Backordered Units            : "
    f"{kpi_balanced['lost_sales_units']:,.0f}"
)

print(
    f"Contract Penalty Exposure    : "
    f"{CONFIG['currency']}"
    f"{kpi_balanced['contract_penalty_exposure']:,.0f}"
)

print(
    f"Service Shortfall Units      : "
    f"{service_shortfall_units:,.0f}"
)


# -----------------------------------------------------------------------------
# POLICY SUMMARY
# -----------------------------------------------------------------------------

balanced_summary = kpis_to_frame(
    kpi_balanced,
    "Balanced Allocation"
)

display(balanced_summary)

**Management Interpretation:** The Balanced Allocation policy evaluates whether a modest reduction in contribution-margin protection can improve customer-service consistency across the portfolio. It is designed for situations where leadership wants to avoid concentrating the shortage on a small number of customers while still protecting enterprise economics.


## Allocation Decision Explainability

Allocation recommendations must be transparent and defensible across Supply Chain, Sales, Finance, and S&OP leadership.

For each order, the system identifies the business factors associated with the allocation decision, including:

- Contribution margin and financial value
- Strategic customer status
- Contractual service commitments
- Product criticality
- Requested delivery timing
- SKU-level supply availability
- Recommended allocation and resulting fill rate

The explainability layer also supports direct order-to-order comparison, helping decision-makers understand the trade-offs behind different allocation outcomes.

This creates a clear **audit trail for management review, commercial escalation, and final allocation approval**.

In [ ]:
# =============================================================================
# ALLOCATION DECISION EXPLAINABILITY
# Order-Level Business Rationale & Comparison
# =============================================================================

def build_allocation_rationale(
    order_df: pd.DataFrame,
    allocation_detail: pd.DataFrame
) -> pd.DataFrame:
    """
    Create an auditable order-level explanation of allocation outcomes.

    The rationale reflects actual order economics, customer commitments,
    delivery urgency, product criticality, SKU supply conditions, and the
    resulting allocation decision.
    """

    rationale = allocation_detail.copy()

    # -------------------------------------------------------------------------
    # SKU SUPPLY CONTEXT
    # -------------------------------------------------------------------------

    supply_context = sku_view[
        [
            "sku_id",
            "demand_units",
            "total_available_supply",
            "shortage_units",
            "shortage_pct",
            "supply_status"
        ]
    ].copy()

    rationale = rationale.merge(
        supply_context,
        on="sku_id",
        how="left"
    )


    # -------------------------------------------------------------------------
    # BUSINESS PRIORITY LABELS
    # -------------------------------------------------------------------------

    rationale["customer_priority"] = rationale["priority_tier"].map({
        1: "Tier 1 - Protected",
        2: "Tier 2 - Priority",
        3: "Tier 3 - Standard"
    })

    rationale["financial_priority"] = pd.qcut(
        rationale["contribution_margin_per_unit"],
        q=3,
        labels=[
            "Lower Margin",
            "Medium Margin",
            "Higher Margin"
        ],
        duplicates="drop"
    )


    # -------------------------------------------------------------------------
    # ALLOCATION OUTCOME
    # -------------------------------------------------------------------------

    rationale["allocation_status"] = np.select(
        [
            rationale["fill_rate"] >= 0.999,
            rationale["fill_rate"] > 0
        ],
        [
            "Fully Allocated",
            "Partially Allocated"
        ],
        default="Deferred / Backordered"
    )


    # -------------------------------------------------------------------------
    # MANAGEMENT RATIONALE
    # -------------------------------------------------------------------------

    def create_rationale(row):

        drivers = []

        if row["strategic_flag"]:
            drivers.append("strategic customer protection")

        if row["has_contract"]:
            drivers.append("contractual service commitment")

        if row["financial_priority"] == "Higher Margin":
            drivers.append("high contribution-margin value")

        if row["criticality"] == "High":
            drivers.append("high product criticality")

        if row["delivery_urgency"] == "Immediate":
            drivers.append("near-term delivery requirement")

        if row["supply_status"] == "Critical Shortage":
            drivers.append("severe SKU supply constraint")

        elif row["supply_status"] == "Constrained":
            drivers.append("SKU supply constraint")

        if not drivers:
            drivers.append("standard commercial demand")

        return "; ".join(drivers)


    rationale["allocation_rationale"] = rationale.apply(
        create_rationale,
        axis=1
    )

    return rationale


# =============================================================================
# BUILD EXPLAINABILITY VIEW
# =============================================================================

allocation_explainability = build_allocation_rationale(
    orders,
    detail_margin
)


# =============================================================================
# SINGLE-ORDER EXPLANATION
# =============================================================================

def explain_order(
    order_id: str,
    explainability_df: pd.DataFrame = allocation_explainability
):
    """
    Display the business rationale and allocation outcome for one customer order.
    """

    result = explainability_df[
        explainability_df["order_id"] == order_id
    ]

    if result.empty:
        print(f"Order {order_id} was not found.")
        return

    row = result.iloc[0]

    print("ORDER ALLOCATION RATIONALE")
    print("-" * 65)

    print(f"Order                    : {row['order_id']}")
    print(f"Customer                 : {row['customer_name']}")
    print(f"Customer Priority        : {row['customer_priority']}")
    print(f"SKU                      : {row['sku_id']}")
    print(f"Product Criticality      : {row['criticality']}")
    print(f"Delivery Urgency         : {row['delivery_urgency']}")

    print("\nSUPPLY POSITION")
    print("-" * 65)

    print(f"SKU Demand               : {row['demand_units']:,.0f} units")
    print(f"Available SKU Supply     : {row['total_available_supply']:,.0f} units")
    print(f"SKU Shortage             : {row['shortage_units']:,.0f} units")
    print(f"SKU Shortage %           : {row['shortage_pct']:.1%}")
    print(f"Supply Status            : {row['supply_status']}")

    print("\nORDER ECONOMICS")
    print("-" * 65)

    print(f"Order Quantity           : {row['order_quantity']:,.0f} units")
    print(f"Order Value              : ${row['order_value']:,.0f}")
    print(
        f"Contribution Margin      : "
        f"${row['contribution_margin_total']:,.0f}"
    )
    print(
        f"Margin per Unit          : "
        f"${row['contribution_margin_per_unit']:,.2f}"
    )

    print("\nALLOCATION DECISION")
    print("-" * 65)

    print(f"Allocated Quantity       : {row['allocated_units']:,.0f} units")
    print(f"Fill Rate                : {row['fill_rate']:.1%}")
    print(f"Allocation Status        : {row['allocation_status']}")
    print(f"Revenue at Risk          : ${row['revenue_at_risk']:,.0f}")
    print(f"Margin at Risk           : ${row['margin_at_risk']:,.0f}")

    print("\nBUSINESS RATIONALE")
    print("-" * 65)

    print(row["allocation_rationale"])


# =============================================================================
# EXAMPLE: FULLY SERVED vs. CONSTRAINED ORDER
# =============================================================================

fully_allocated = allocation_explainability[
    allocation_explainability["fill_rate"] >= 0.999
]

constrained_orders = allocation_explainability[
    allocation_explainability["fill_rate"] < 0.999
]

if not fully_allocated.empty:

    example_full = (
        fully_allocated
        .sort_values(
            "contribution_margin_total",
            ascending=False
        )
        .iloc[0]["order_id"]
    )

    print("=== FULLY ALLOCATED ORDER EXAMPLE ===\n")
    explain_order(example_full)


if not constrained_orders.empty:

    example_constrained = (
        constrained_orders
        .sort_values(
            "margin_at_risk",
            ascending=False
        )
        .iloc[0]["order_id"]
    )

    print("\n\n=== CONSTRAINED ORDER EXAMPLE ===\n")
    explain_order(example_constrained)

## Allocation Policy Comparison

The five allocation strategies are evaluated using the same financial, service, and customer-priority KPI framework to identify the strongest management policy for the constrained supply period.

Policies compared:

- **FIFO** — prioritizes orders by entry sequence
- **Proportional Fair Share** — distributes supply evenly within each SKU
- **Margin Protection** — prioritizes contribution-margin preservation
- **Strategic Customer Protection** — places greater emphasis on strategic-account service
- **Balanced Allocation** — balances margin protection with customer-service fairness

The comparison focuses on the trade-offs among **fill rate, strategic customer service, contribution margin protected, revenue-at-risk, margin-at-risk, backorders, and contract penalty exposure**.

The objective is to determine which policy provides the strongest overall business outcome—not simply the highest fulfillment rate.

In [ ]:
# =============================================================================
# EXECUTIVE POLICY COMPARISON | BASE CASE
# =============================================================================

policy_results = pd.concat(
    [
        kpis_to_frame(
            kpi_fifo,
            "FIFO"
        ),
        kpis_to_frame(
            kpi_prop,
            "Proportional Fair Share"
        ),
        kpis_to_frame(
            kpi_margin,
            "Margin Protection"
        ),
        kpis_to_frame(
            kpi_strategic,
            "Strategic Customer Protection"
        ),
        kpis_to_frame(
            kpi_balanced,
            "Balanced Allocation"
        ),
    ],
    ignore_index=True
)


# -----------------------------------------------------------------------------
# SELECT MANAGEMENT KPIs
# -----------------------------------------------------------------------------

comparison_columns = [
    "policy",
    "fill_rate",
    "otif_rate",
    "strategic_customer_service_level",
    "contract_customer_service_level",
    "contribution_margin_protected",
    "revenue_at_risk",
    "margin_at_risk",
    "contract_penalty_exposure",
    "lost_sales_units",
    "perfect_order_rate"
]

policy_comparison = policy_results[
    comparison_columns
].copy()


# -----------------------------------------------------------------------------
# FORMAT PERCENTAGE KPIs
# -----------------------------------------------------------------------------

percentage_columns = [
    "fill_rate",
    "otif_rate",
    "strategic_customer_service_level",
    "contract_customer_service_level",
    "perfect_order_rate"
]

for column in percentage_columns:
    policy_comparison[column] = (
        policy_comparison[column] * 100
    ).round(1)


# -----------------------------------------------------------------------------
# FORMAT FINANCIAL KPIs
# -----------------------------------------------------------------------------

financial_columns = [
    "contribution_margin_protected",
    "revenue_at_risk",
    "margin_at_risk",
    "contract_penalty_exposure"
]

for column in financial_columns:
    policy_comparison[column] = (
        policy_comparison[column]
        .round(0)
        .astype(int)
    )


# -----------------------------------------------------------------------------
# FORMAT OPERATIONAL KPIs
# -----------------------------------------------------------------------------

policy_comparison["lost_sales_units"] = (
    policy_comparison["lost_sales_units"]
    .round(0)
    .astype(int)
)


# -----------------------------------------------------------------------------
# EXECUTIVE COLUMN LABELS
# -----------------------------------------------------------------------------

policy_comparison.columns = [
    "Policy",
    "Fill Rate %",
    "OTIF %",
    "Strategic Service %",
    "Contract Service %",
    "Margin Protected $",
    "Revenue at Risk $",
    "Margin at Risk $",
    "Contract Penalty Exposure $",
    "Backordered Units",
    "Perfect Order %"
]


# -----------------------------------------------------------------------------
# RANK POLICIES BY CONTRIBUTION MARGIN PROTECTION
# -----------------------------------------------------------------------------

policy_comparison["Margin Protection Rank"] = (
    policy_comparison["Margin Protected $"]
    .rank(
        ascending=False,
        method="dense"
    )
    .astype(int)
)


# -----------------------------------------------------------------------------
# DISPLAY EXECUTIVE COMPARISON
# -----------------------------------------------------------------------------

policy_comparison = (
    policy_comparison
    .sort_values(
        "Margin Protection Rank"
    )
    .reset_index(drop=True)
)

display(
    policy_comparison.style
    .format({
        "Fill Rate %": "{:.1f}%",
        "OTIF %": "{:.1f}%",
        "Strategic Service %": "{:.1f}%",
        "Contract Service %": "{:.1f}%",
        "Margin Protected $": "${:,.0f}",
        "Revenue at Risk $": "${:,.0f}",
        "Margin at Risk $": "${:,.0f}",
        "Contract Penalty Exposure $": "${:,.0f}",
        "Backordered Units": "{:,.0f}",
        "Perfect Order %": "{:.1f}%"
    })
)

In [ ]:
# =============================================================================
# EXECUTIVE POLICY COMPARISON | Financial Outcomes
# =============================================================================

# -----------------------------------------------------------------------------
# FINANCIAL COMPARISON VIEW
# -----------------------------------------------------------------------------

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Contribution Margin Protected",
        "Revenue at Risk"
    )
)

fig.add_trace(
    go.Bar(
        x=policy_results["policy"],
        y=policy_results["contribution_margin_protected"],
        name="Margin Protected",
        showlegend=False
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Bar(
        x=policy_results["policy"],
        y=policy_results["revenue_at_risk"],
        name="Revenue at Risk",
        showlegend=False
    ),
    row=1,
    col=2
)

fig.update_layout(
    height=450,
    title_text="Financial Performance by Allocation Policy — Base Case"
)

fig.update_yaxes(
    title_text="Financial Value ($)",
    tickprefix="$",
    tickformat="~s",
    row=1,
    col=1
)

fig.update_yaxes(
    title_text="Financial Exposure ($)",
    tickprefix="$",
    tickformat="~s",
    row=1,
    col=2
)

fig.update_xaxes(
    tickangle=-25
)

fig.show()


# -----------------------------------------------------------------------------
# FINANCIAL PERFORMANCE SUMMARY
# -----------------------------------------------------------------------------

best_margin_idx = (
    policy_results["contribution_margin_protected"]
    .idxmax()
)

best_margin_policy = policy_results.loc[
    best_margin_idx,
    "policy"
]

best_margin_value = float(
    policy_results.loc[
        best_margin_idx,
        "contribution_margin_protected"
    ]
)

fifo_margin = float(
    policy_results.loc[
        policy_results["policy"] == "FIFO",
        "contribution_margin_protected"
    ].iloc[0]
)

margin_improvement = (
    best_margin_value - fifo_margin
)

margin_improvement_pct = (
    margin_improvement / fifo_margin
    if fifo_margin > 0
    else 0
)

print("FINANCIAL POLICY ASSESSMENT")
print("-" * 60)

print(
    f"Highest Margin Protection : "
    f"{best_margin_policy}"
)

print(
    f"Protected Margin          : "
    f"{CONFIG['currency']}"
    f"{best_margin_value:,.0f}"
)

print(
    f"Improvement vs. FIFO      : "
    f"{CONFIG['currency']}"
    f"{margin_improvement:,.0f}"
)

print(
    f"Relative Improvement      : "
    f"{margin_improvement_pct:.1%}"
)



### Financial Policy Assessment

The base-case comparison quantifies the financial impact of each allocation strategy under the same constrained supply conditions.

The analysis focuses on two primary financial outcomes:

- **Contribution Margin Protected** — economic value preserved through fulfilled customer demand.
- **Revenue at Risk** — confirmed order value remaining exposed because available supply cannot satisfy total demand.

Performance is measured relative to **FIFO**, providing a clear view of the incremental financial value created by more targeted allocation strategies.

**Management Interpretation:** Financial performance alone does not determine the final allocation decision. The preferred policy must also maintain an acceptable balance across **strategic-customer service, contractual commitments, customer fairness, and overall supply constraints**.

## Customer Service Trade-Off Assessment

Optimization improves the use of constrained supply, but the financial benefit may result in different service outcomes across customers and orders.

This assessment identifies where the **Margin Protection policy changes allocation relative to FIFO and Proportional Fair Share**, with particular attention to customers receiving lower fulfillment levels.

The review focuses on:

- Customers and orders experiencing reduced allocation
- Units and revenue shifted between allocation policies
- Strategic and contractual accounts affected
- Contribution margin associated with the reallocation
- SKU-level supply constraints driving the service trade-off
- Resulting customer fill-rate differences

**Management Objective:** Make the allocation trade-offs fully visible before approval, ensuring that incremental financial value is evaluated alongside customer relationships, contractual commitments, and service risk.

In [ ]:
# =============================================================================
# CUSTOMER SERVICE TRADE-OFF | FIFO vs. MARGIN PROTECTION
# =============================================================================

# -----------------------------------------------------------------------------
# FIFO CUSTOMER SERVICE
# -----------------------------------------------------------------------------

customer_fifo = (
    detail_fifo
    .groupby("customer_id", as_index=False)
    .agg(
        fifo_demand_units=("order_quantity", "sum"),
        fifo_allocated_units=("allocated_units", "sum")
    )
)

customer_fifo["fifo_fill_rate"] = np.where(
    customer_fifo["fifo_demand_units"] > 0,
    customer_fifo["fifo_allocated_units"]
    / customer_fifo["fifo_demand_units"],
    0
)


# -----------------------------------------------------------------------------
# MARGIN PROTECTION CUSTOMER SERVICE
# -----------------------------------------------------------------------------

customer_margin = (
    detail_margin
    .groupby("customer_id", as_index=False)
    .agg(
        optimized_demand_units=("order_quantity", "sum"),
        optimized_allocated_units=("allocated_units", "sum")
    )
)

customer_margin["optimized_fill_rate"] = np.where(
    customer_margin["optimized_demand_units"] > 0,
    customer_margin["optimized_allocated_units"]
    / customer_margin["optimized_demand_units"],
    0
)


# -----------------------------------------------------------------------------
# CUSTOMER PORTFOLIO COMPARISON
# -----------------------------------------------------------------------------

customer_tradeoff = (
    customer_fifo
    .merge(
        customer_margin,
        on="customer_id",
        how="inner"
    )
    .merge(
        customers[
            [
                "customer_id",
                "customer_name",
                "customer_segment",
                "customer_tier",
                "strategic_flag",
                "has_contract"
            ]
        ],
        on="customer_id",
        how="left"
    )
)

customer_tradeoff["fill_rate_change_pp"] = (
    (
        customer_tradeoff["optimized_fill_rate"]
        - customer_tradeoff["fifo_fill_rate"]
    ) * 100
).round(1)

customer_tradeoff["allocation_change_units"] = (
    customer_tradeoff["optimized_allocated_units"]
    - customer_tradeoff["fifo_allocated_units"]
).round(0)


# -----------------------------------------------------------------------------
# CLASSIFY SERVICE IMPACT
# -----------------------------------------------------------------------------

customer_tradeoff["service_impact"] = np.select(
    [
        customer_tradeoff["fill_rate_change_pp"] > 1,
        customer_tradeoff["fill_rate_change_pp"] < -1
    ],
    [
        "Service Increase",
        "Service Reduction"
    ],
    default="Minimal Change"
)


# -----------------------------------------------------------------------------
# MANAGEMENT SUMMARY
# -----------------------------------------------------------------------------

customers_reduced = (
    customer_tradeoff["service_impact"]
    == "Service Reduction"
).sum()

customers_improved = (
    customer_tradeoff["service_impact"]
    == "Service Increase"
).sum()

customers_unchanged = (
    customer_tradeoff["service_impact"]
    == "Minimal Change"
).sum()

print("CUSTOMER SERVICE TRADE-OFF")
print("-" * 60)

print(f"Customers with Service Reduction : {customers_reduced}")
print(f"Customers with Service Increase  : {customers_improved}")
print(f"Customers with Minimal Change    : {customers_unchanged}")


# -----------------------------------------------------------------------------
# CUSTOMERS WITH LARGEST SERVICE REDUCTIONS
# -----------------------------------------------------------------------------

service_reductions = (
    customer_tradeoff[
        customer_tradeoff["service_impact"]
        == "Service Reduction"
    ]
    .sort_values(
        "fill_rate_change_pp",
        ascending=True
    )
    .head(10)
)

print("\nLARGEST SERVICE REDUCTIONS")
print("-" * 60)

display(
    service_reductions[
        [
            "customer_id",
            "customer_name",
            "customer_segment",
            "customer_tier",
            "strategic_flag",
            "has_contract",
            "fifo_fill_rate",
            "optimized_fill_rate",
            "fill_rate_change_pp",
            "allocation_change_units"
        ]
    ].style.format({
        "fifo_fill_rate": "{:.1%}",
        "optimized_fill_rate": "{:.1%}",
        "fill_rate_change_pp": "{:+.1f} pp",
        "allocation_change_units": "{:+,.0f}"
    })
)


# -----------------------------------------------------------------------------
# CUSTOMERS WITH LARGEST SERVICE GAINS
# -----------------------------------------------------------------------------

service_gains = (
    customer_tradeoff[
        customer_tradeoff["service_impact"]
        == "Service Increase"
    ]
    .sort_values(
        "fill_rate_change_pp",
        ascending=False
    )
    .head(10)
)

print("\nLARGEST SERVICE GAINS")
print("-" * 60)

display(
    service_gains[
        [
            "customer_id",
            "customer_name",
            "customer_segment",
            "customer_tier",
            "strategic_flag",
            "has_contract",
            "fifo_fill_rate",
            "optimized_fill_rate",
            "fill_rate_change_pp",
            "allocation_change_units"
        ]
    ].style.format({
        "fifo_fill_rate": "{:.1%}",
        "optimized_fill_rate": "{:.1%}",
        "fill_rate_change_pp": "{:+.1f} pp",
        "allocation_change_units": "{:+,.0f}"
    })
)

In [ ]:
# =============================================================================
# CUSTOMERS WITH LARGEST SERVICE GAINS
# Margin Protection vs. FIFO
# =============================================================================

service_gains = (
    customer_tradeoff[
        customer_tradeoff["service_impact"] == "Service Increase"
    ]
    .sort_values(
        "fill_rate_change_pp",
        ascending=False
    )
    .head(10)
)

print("LARGEST CUSTOMER SERVICE GAINS")
print("-" * 60)

display(
    service_gains[
        [
            "customer_id",
            "customer_name",
            "customer_segment",
            "customer_tier",
            "strategic_flag",
            "has_contract",
            "fifo_fill_rate",
            "optimized_fill_rate",
            "fill_rate_change_pp",
            "allocation_change_units"
        ]
    ].style.format({
        "fifo_fill_rate": "{:.1%}",
        "optimized_fill_rate": "{:.1%}",
        "fill_rate_change_pp": "{:+.1f} pp",
        "allocation_change_units": "{:+,.0f}"
    })
)

In [ ]:
# =============================================================================
# CUSTOMER ALLOCATION MATRIX | Margin Protection Policy
# =============================================================================

def customer_allocation_matrix(
    detail_df: pd.DataFrame
) -> pd.DataFrame:

    # -------------------------------------------------------------------------
    # CUSTOMER-LEVEL ALLOCATION SUMMARY
    # -------------------------------------------------------------------------

    matrix = (
        detail_df
        .groupby("customer_id", as_index=False)
        .agg(
            demand_units=("order_quantity", "sum"),
            allocated_units=("allocated_units", "sum"),
            revenue_protected=("revenue_protected", "sum"),
            margin_protected=("margin_protected", "sum"),
            revenue_at_risk=("revenue_at_risk", "sum"),
            margin_at_risk=("margin_at_risk", "sum")
        )
    )

    # -------------------------------------------------------------------------
    # CUSTOMER SERVICE LEVEL
    # -------------------------------------------------------------------------

    matrix["fill_rate"] = np.where(
        matrix["demand_units"] > 0,
        matrix["allocated_units"] / matrix["demand_units"],
        0
    )

    # -------------------------------------------------------------------------
    # CUSTOMER MASTER ATTRIBUTES
    # -------------------------------------------------------------------------

    matrix = matrix.merge(
        customers[
            [
                "customer_id",
                "customer_name",
                "customer_segment",
                "customer_tier",
                "strategic_flag",
                "has_contract"
            ]
        ],
        on="customer_id",
        how="left"
    )

    # -------------------------------------------------------------------------
    # SERVICE STATUS
    # -------------------------------------------------------------------------

    matrix["service_status"] = np.select(
        [
            matrix["fill_rate"] >= 0.98,
            matrix["fill_rate"] >= 0.90,
            matrix["fill_rate"] > 0
        ],
        [
            "Protected",
            "Watch",
            "At Risk"
        ],
        default="Unfulfilled"
    )

    # -------------------------------------------------------------------------
    # EXECUTIVE SORTING
    # -------------------------------------------------------------------------

    tier_order = {
        "Tier 1 - Strategic Contract": 1,
        "Tier 1 - Strategic": 2,
        "Tier 2 - Contract": 3,
        "Tier 3 - Standard": 4
    }

    matrix["tier_sort"] = (
        matrix["customer_tier"]
        .map(tier_order)
        .fillna(99)
    )

    matrix = (
        matrix
        .sort_values(
            [
                "tier_sort",
                "fill_rate",
                "margin_at_risk"
            ],
            ascending=[
                True,
                True,
                False
            ]
        )
        .reset_index(drop=True)
    )

    # -------------------------------------------------------------------------
    # FINAL EXECUTIVE VIEW
    # -------------------------------------------------------------------------

    matrix = matrix[
        [
            "customer_id",
            "customer_name",
            "customer_segment",
            "customer_tier",
            "demand_units",
            "allocated_units",
            "fill_rate",
            "margin_protected",
            "revenue_at_risk",
            "margin_at_risk",
            "service_status"
        ]
    ]

    matrix.columns = [
        "Customer ID",
        "Customer",
        "Segment",
        "Customer Tier",
        "Demand Units",
        "Allocated Units",
        "Fill Rate",
        "Margin Protected",
        "Revenue at Risk",
        "Margin at Risk",
        "Service Status"
    ]

    return matrix


# =============================================================================
# BUILD & DISPLAY CUSTOMER ALLOCATION MATRIX
# =============================================================================

allocation_matrix = customer_allocation_matrix(
    detail_margin
)

print("CUSTOMER ALLOCATION MATRIX — MARGIN PROTECTION POLICY")
print("-" * 70)

display(
    allocation_matrix.head(20).style.format({
        "Demand Units": "{:,.0f}",
        "Allocated Units": "{:,.0f}",
        "Fill Rate": "{:.1%}",
        "Margin Protected": "${:,.0f}",
        "Revenue at Risk": "${:,.0f}",
        "Margin at Risk": "${:,.0f}"
    })
)

## Financial Impact Assessment

The financial impact framework translates allocation-policy differences into measurable business value.

The analysis quantifies:

- Incremental contribution margin protected
- Reduction in revenue-at-risk
- Reduction in margin-at-risk
- Change in contract-penalty exposure
- Financial value created versus the FIFO baseline
- Estimated return on the decision-support process

The objective is to show whether a more targeted allocation strategy creates sufficient financial value to justify its implementation and ongoing planning effort.

> **Financial Reporting Note:** All financial outcomes are modeled using the simulated enterprise order book and should be interpreted as scenario-based estimates rather than actual company results.

In [ ]:
# =============================================================================
# FINANCIAL IMPACT ASSESSMENT
# Proposed Allocation Policies vs. FIFO Baseline
# =============================================================================

# Modeled annual cost to maintain the decision-support process
IMPLEMENTATION_COST_ESTIMATE = 45_000

# Illustrative assumption:
# three material constrained-supply events of comparable scale per year
CONSTRAINED_PERIODS_PER_YEAR = 3


def financial_impact(
    baseline_kpi: dict,
    proposed_kpi: dict,
    policy_name: str
) -> pd.DataFrame:
    """
    Quantify the modeled financial value of a proposed allocation policy
    relative to the FIFO baseline.
    """

    # -------------------------------------------------------------------------
    # PERIOD-LEVEL FINANCIAL IMPACT
    # -------------------------------------------------------------------------

    margin_gain = (
        proposed_kpi["contribution_margin_protected"]
        - baseline_kpi["contribution_margin_protected"]
    )

    revenue_at_risk_reduction = (
        baseline_kpi["revenue_at_risk"]
        - proposed_kpi["revenue_at_risk"]
    )

    margin_at_risk_reduction = (
        baseline_kpi["margin_at_risk"]
        - proposed_kpi["margin_at_risk"]
    )

    penalty_reduction = (
        baseline_kpi["contract_penalty_exposure"]
        - proposed_kpi["contract_penalty_exposure"]
    )


    # -------------------------------------------------------------------------
    # ANNUALIZED VALUE
    # -------------------------------------------------------------------------

    # Avoid double counting revenue-at-risk reduction because incremental
    # contribution margin already represents the primary economic benefit.
    period_financial_benefit = (
        margin_gain
        + penalty_reduction
    )

    modeled_annual_benefit = (
        period_financial_benefit
        * CONSTRAINED_PERIODS_PER_YEAR
    )

    net_annual_benefit = (
        modeled_annual_benefit
        - IMPLEMENTATION_COST_ESTIMATE
    )


    # -------------------------------------------------------------------------
    # INVESTMENT METRICS
    # -------------------------------------------------------------------------

    benefit_cost_ratio = (
        modeled_annual_benefit
        / IMPLEMENTATION_COST_ESTIMATE
        if IMPLEMENTATION_COST_ESTIMATE > 0
        else np.nan
    )

    roi_pct = (
        net_annual_benefit
        / IMPLEMENTATION_COST_ESTIMATE
        if IMPLEMENTATION_COST_ESTIMATE > 0
        else np.nan
    )

    payback_years = (
        IMPLEMENTATION_COST_ESTIMATE
        / modeled_annual_benefit
        if modeled_annual_benefit > 0
        else np.nan
    )

    payback_months = (
        payback_years * 12
        if pd.notna(payback_years)
        else np.nan
    )


    # -------------------------------------------------------------------------
    # FINANCIAL SUMMARY
    # -------------------------------------------------------------------------

    return pd.DataFrame([{
        "Policy": policy_name,

        "Incremental Margin Protected / Period":
            margin_gain,

        "Revenue-at-Risk Reduction / Period":
            revenue_at_risk_reduction,

        "Margin-at-Risk Reduction / Period":
            margin_at_risk_reduction,

        "Contract Penalty Reduction / Period":
            penalty_reduction,

        "Modeled Annual Benefit":
            modeled_annual_benefit,

        "Annual Implementation Cost":
            IMPLEMENTATION_COST_ESTIMATE,

        "Net Annual Benefit":
            net_annual_benefit,

        "Benefit-Cost Ratio":
            benefit_cost_ratio,

        "Modeled ROI":
            roi_pct,

        "Payback Months":
            payback_months
    }])


# =============================================================================
# POLICY FINANCIAL ASSESSMENT
# =============================================================================

fin_margin = financial_impact(
    kpi_fifo,
    kpi_margin,
    "Margin Protection"
)

fin_balanced = financial_impact(
    kpi_fifo,
    kpi_balanced,
    "Balanced Allocation"
)

fin_strategic = financial_impact(
    kpi_fifo,
    kpi_strategic,
    "Strategic Customer Protection"
)

financial_summary = pd.concat(
    [
        fin_margin,
        fin_balanced,
        fin_strategic
    ],
    ignore_index=True
)


# =============================================================================
# EXECUTIVE FINANCIAL VIEW
# =============================================================================

display(
    financial_summary.style.format({
        "Incremental Margin Protected / Period": "${:,.0f}",
        "Revenue-at-Risk Reduction / Period": "${:,.0f}",
        "Margin-at-Risk Reduction / Period": "${:,.0f}",
        "Contract Penalty Reduction / Period": "${:,.0f}",
        "Modeled Annual Benefit": "${:,.0f}",
        "Annual Implementation Cost": "${:,.0f}",
        "Net Annual Benefit": "${:,.0f}",
        "Benefit-Cost Ratio": "{:.2f}x",
        "Modeled ROI": "{:.1%}",
        "Payback Months": "{:.1f}"
    })
)

# Baseline vs. Proposed Policy — Management Comparison

This comparison provides a consolidated view of the current-state **FIFO allocation approach** against the proposed **Balanced Allocation policy**.

The assessment brings together the financial, operational, and customer-service measures required for an S&OP allocation decision, including:

* Overall fill rate and OTIF performance
* Strategic and contracted customer service levels
* Contribution margin protected
* Revenue and margin at risk
* Contract penalty exposure
* Backordered demand
* Customer-service trade-offs

**Management Objective:** Determine whether Balanced Allocation provides sufficient improvement in financial protection and customer-service consistency to justify replacing the FIFO baseline for constrained-supply decisions.

The comparison supports final policy selection by making the economic benefit, service impact, and customer-risk trade-offs visible in a single management view.


In [ ]:
# =============================================================================
# BASELINE vs. PROPOSED POLICY | MANAGEMENT COMPARISON
# FIFO vs. Balanced Allocation
# =============================================================================

def fmt_pct(value):
    return f"{value:.1%}"


def fmt_usd(value):
    return f"{CONFIG['currency']}{value:,.0f}"


def fmt_units(value):
    return f"{value:,.0f}"


def fmt_pp(value):
    return f"{value * 100:+.1f} pp"


# -----------------------------------------------------------------------------
# BUILD MANAGEMENT COMPARISON
# -----------------------------------------------------------------------------

comparison_rows = [

    {
        "KPI": "Fill Rate",
        "Current State (FIFO)": fmt_pct(kpi_fifo["fill_rate"]),
        "Proposed (Balanced)": fmt_pct(kpi_balanced["fill_rate"]),
        "Change": fmt_pp(
            kpi_balanced["fill_rate"]
            - kpi_fifo["fill_rate"]
        ),
        "Business Impact": "Improved" if
        kpi_balanced["fill_rate"] > kpi_fifo["fill_rate"]
        else "Unchanged" if
        np.isclose(
            kpi_balanced["fill_rate"],
            kpi_fifo["fill_rate"]
        )
        else "Reduced"
    },

    {
        "KPI": "OTIF",
        "Current State (FIFO)": fmt_pct(kpi_fifo["otif_rate"]),
        "Proposed (Balanced)": fmt_pct(kpi_balanced["otif_rate"]),
        "Change": fmt_pp(
            kpi_balanced["otif_rate"]
            - kpi_fifo["otif_rate"]
        ),
        "Business Impact": "Improved" if
        kpi_balanced["otif_rate"] > kpi_fifo["otif_rate"]
        else "Unchanged" if
        np.isclose(
            kpi_balanced["otif_rate"],
            kpi_fifo["otif_rate"]
        )
        else "Reduced"
    },

    {
        "KPI": "Strategic Customer Service",
        "Current State (FIFO)": fmt_pct(
            kpi_fifo["strategic_customer_service_level"]
        ),
        "Proposed (Balanced)": fmt_pct(
            kpi_balanced["strategic_customer_service_level"]
        ),
        "Change": fmt_pp(
            kpi_balanced["strategic_customer_service_level"]
            - kpi_fifo["strategic_customer_service_level"]
        ),
        "Business Impact": "Improved" if
        kpi_balanced["strategic_customer_service_level"]
        > kpi_fifo["strategic_customer_service_level"]
        else "Unchanged" if
        np.isclose(
            kpi_balanced["strategic_customer_service_level"],
            kpi_fifo["strategic_customer_service_level"]
        )
        else "Reduced"
    },

    {
        "KPI": "Contract Customer Service",
        "Current State (FIFO)": fmt_pct(
            kpi_fifo["contract_customer_service_level"]
        ),
        "Proposed (Balanced)": fmt_pct(
            kpi_balanced["contract_customer_service_level"]
        ),
        "Change": fmt_pp(
            kpi_balanced["contract_customer_service_level"]
            - kpi_fifo["contract_customer_service_level"]
        ),
        "Business Impact": "Improved" if
        kpi_balanced["contract_customer_service_level"]
        > kpi_fifo["contract_customer_service_level"]
        else "Unchanged" if
        np.isclose(
            kpi_balanced["contract_customer_service_level"],
            kpi_fifo["contract_customer_service_level"]
        )
        else "Reduced"
    },

    {
        "KPI": "Contribution Margin Protected",
        "Current State (FIFO)": fmt_usd(
            kpi_fifo["contribution_margin_protected"]
        ),
        "Proposed (Balanced)": fmt_usd(
            kpi_balanced["contribution_margin_protected"]
        ),
        "Change": fmt_usd(
            kpi_balanced["contribution_margin_protected"]
            - kpi_fifo["contribution_margin_protected"]
        ),
        "Business Impact": "Improved" if
        kpi_balanced["contribution_margin_protected"]
        > kpi_fifo["contribution_margin_protected"]
        else "Reduced"
    },

    {
        "KPI": "Revenue at Risk",
        "Current State (FIFO)": fmt_usd(
            kpi_fifo["revenue_at_risk"]
        ),
        "Proposed (Balanced)": fmt_usd(
            kpi_balanced["revenue_at_risk"]
        ),
        "Change": fmt_usd(
            kpi_balanced["revenue_at_risk"]
            - kpi_fifo["revenue_at_risk"]
        ),
        "Business Impact": "Improved" if
        kpi_balanced["revenue_at_risk"]
        < kpi_fifo["revenue_at_risk"]
        else "Reduced"
    },

    {
        "KPI": "Margin at Risk",
        "Current State (FIFO)": fmt_usd(
            kpi_fifo["margin_at_risk"]
        ),
        "Proposed (Balanced)": fmt_usd(
            kpi_balanced["margin_at_risk"]
        ),
        "Change": fmt_usd(
            kpi_balanced["margin_at_risk"]
            - kpi_fifo["margin_at_risk"]
        ),
        "Business Impact": "Improved" if
        kpi_balanced["margin_at_risk"]
        < kpi_fifo["margin_at_risk"]
        else "Reduced"
    },

    {
        "KPI": "Contract Penalty Exposure",
        "Current State (FIFO)": fmt_usd(
            kpi_fifo["contract_penalty_exposure"]
        ),
        "Proposed (Balanced)": fmt_usd(
            kpi_balanced["contract_penalty_exposure"]
        ),
        "Change": fmt_usd(
            kpi_balanced["contract_penalty_exposure"]
            - kpi_fifo["contract_penalty_exposure"]
        ),
        "Business Impact": "Improved" if
        kpi_balanced["contract_penalty_exposure"]
        < kpi_fifo["contract_penalty_exposure"]
        else "Reduced"
    },

    {
        "KPI": "Backordered Units",
        "Current State (FIFO)": fmt_units(
            kpi_fifo["lost_sales_units"]
        ),
        "Proposed (Balanced)": fmt_units(
            kpi_balanced["lost_sales_units"]
        ),
        "Change": fmt_units(
            kpi_balanced["lost_sales_units"]
            - kpi_fifo["lost_sales_units"]
        ),
        "Business Impact": "Improved" if
        kpi_balanced["lost_sales_units"]
        < kpi_fifo["lost_sales_units"]
        else "Unchanged" if
        np.isclose(
            kpi_balanced["lost_sales_units"],
            kpi_fifo["lost_sales_units"]
        )
        else "Reduced"
    },

    {
        "KPI": "Perfect Order Rate",
        "Current State (FIFO)": fmt_pct(
            kpi_fifo["perfect_order_rate"]
        ),
        "Proposed (Balanced)": fmt_pct(
            kpi_balanced["perfect_order_rate"]
        ),
        "Change": fmt_pp(
            kpi_balanced["perfect_order_rate"]
            - kpi_fifo["perfect_order_rate"]
        ),
        "Business Impact": "Improved" if
        kpi_balanced["perfect_order_rate"]
        > kpi_fifo["perfect_order_rate"]
        else "Unchanged" if
        np.isclose(
            kpi_balanced["perfect_order_rate"],
            kpi_fifo["perfect_order_rate"]
        )
        else "Reduced"
    }
]


# -----------------------------------------------------------------------------
# FINAL MANAGEMENT TABLE
# -----------------------------------------------------------------------------

baseline_vs_proposed = pd.DataFrame(comparison_rows)

print("BASELINE vs. PROPOSED POLICY — MANAGEMENT COMPARISON")
print("-" * 75)

display(
    baseline_vs_proposed.style
    .set_properties(**{
        "text-align": "left"
    })
)

## Scenario & Stress Testing

The allocation strategy is evaluated across a range of supply, demand, and customer-priority conditions to determine how effectively it performs as operating constraints change.

Five scenarios are assessed:

1. **Base Case** — current constrained-supply environment based on the modeled order book and available capacity.
2. **Demand Surge** — confirmed demand increases by 15% while available supply remains unchanged.
3. **Deeper Supply Disruption** — available supply declines by an additional 20%, representing further production or supplier disruption.
4. **Strategic Customer Protection** — the strategic-customer service target increases to 98%, testing the financial impact of stronger account protection.
5. **Substitution Recovery** — approved product substitution is introduced for selected constrained SKUs to measure potential service and financial recovery.

Each scenario compares **FIFO and Balanced Allocation** using a consistent KPI framework across:

- Overall fill rate and backordered demand
- Strategic and contracted customer service
- Contribution margin protected
- Revenue and margin at risk
- Contract penalty exposure
- Supply-recovery opportunity

**Management Objective:** Assess the resilience of the Balanced Allocation policy, identify conditions that materially change its business value, and determine when additional actions—such as supply recovery, substitution, or revised customer-service priorities—are required.

In [ ]:
# =============================================================================
# SCENARIO & STRESS-TEST ENGINE
# FIFO vs. Balanced Allocation
# =============================================================================

def rebuild_scenario(
    demand_multiplier: float = 1.0,
    supply_multiplier: float = 1.0,
    substitution_source_sku: str = None,
    substitution_redirect_pct: float = 0.40
):
    """
    Rebuild the order book and SKU supply position for a scenario.

    Parameters
    ----------
    demand_multiplier : float
        Multiplier applied to confirmed customer demand.

    supply_multiplier : float
        Multiplier applied to available production.
        On-hand inventory is preserved.

    substitution_source_sku : str or None
        Constrained source SKU receiving approved substitution recovery.

    substitution_redirect_pct : float
        Maximum share of the substitute SKU's SURPLUS supply that may
        be redirected to support the constrained source SKU.

    Returns
    -------
    scenario_orders : pd.DataFrame
    scenario_supply : pd.DataFrame
    substitution_recovered : float
        Source-SKU-equivalent units recovered through substitution.
    """

    # -------------------------------------------------------------------------
    # 1. SCENARIO ORDER BOOK
    # -------------------------------------------------------------------------

    scenario_orders = orders.copy()

    scenario_orders["order_quantity"] = (
        scenario_orders["order_quantity"]
        * demand_multiplier
    ).round().clip(lower=1).astype(int)

    scenario_orders["order_value"] = (
        scenario_orders["selling_price"]
        * scenario_orders["order_quantity"]
    ).round(2)

    scenario_orders["contribution_margin_total"] = (
        scenario_orders["contribution_margin_per_unit"]
        * scenario_orders["order_quantity"]
    ).round(2)


    # -------------------------------------------------------------------------
    # 2. SCENARIO SUPPLY POSITION
    # -------------------------------------------------------------------------

    scenario_supply = supply_by_sku.copy()

    scenario_supply["available_production"] = (
        scenario_supply["available_production"]
        * supply_multiplier
    ).round().clip(lower=0)

    scenario_supply["total_available_supply"] = (
        scenario_supply["available_production"]
        + scenario_supply["available_inventory"]
    )


    # -------------------------------------------------------------------------
    # 3. SCENARIO DEMAND BY SKU
    # Build demand before substitution so donor surplus and recipient shortage
    # can be measured correctly.
    # -------------------------------------------------------------------------

    scenario_demand = (
        scenario_orders
        .groupby("sku_id", as_index=False)
        .agg(
            demand_units=("order_quantity", "sum")
        )
    )

    demand_map = (
        scenario_demand
        .set_index("sku_id")["demand_units"]
        .to_dict()
    )


    # -------------------------------------------------------------------------
    # 4. APPROVED SUBSTITUTION RECOVERY
    #
    # Business rule:
    # - The donor SKU must satisfy its own demand first.
    # - Only genuine donor SURPLUS may be redirected.
    # - At most substitution_redirect_pct of donor surplus is eligible.
    # - Recovery cannot exceed the constrained source SKU's actual shortage.
    #
    # This prevents substitution from fixing one SKU by creating a larger
    # shortage on another SKU.
    # -------------------------------------------------------------------------

    substitution_recovered = 0.0

    if substitution_source_sku is not None:

        route = substitutions[
            substitutions["source_sku"]
            == substitution_source_sku
        ]

        if not route.empty:

            substitute_sku = route.iloc[0]["substitute_sku"]

            conversion_ratio = float(
                route.iloc[0]["conversion_ratio"]
            )

            donor_mask = (
                scenario_supply["sku_id"]
                == substitute_sku
            )

            recipient_mask = (
                scenario_supply["sku_id"]
                == substitution_source_sku
            )

            if (
                donor_mask.any()
                and recipient_mask.any()
                and conversion_ratio > 0
            ):

                # -------------------------------------------------------------
                # Donor SKU position
                # -------------------------------------------------------------

                donor_supply = float(
                    scenario_supply.loc[
                        donor_mask,
                        "total_available_supply"
                    ].iloc[0]
                )

                donor_demand = float(
                    demand_map.get(
                        substitute_sku,
                        0
                    )
                )

                donor_surplus = max(
                    donor_supply - donor_demand,
                    0.0
                )


                # -------------------------------------------------------------
                # Recipient SKU position
                # -------------------------------------------------------------

                recipient_supply = float(
                    scenario_supply.loc[
                        recipient_mask,
                        "total_available_supply"
                    ].iloc[0]
                )

                recipient_demand = float(
                    demand_map.get(
                        substitution_source_sku,
                        0
                    )
                )

                recipient_shortage = max(
                    recipient_demand
                    - recipient_supply,
                    0.0
                )


                # -------------------------------------------------------------
                # Eligible donor quantity
                # -------------------------------------------------------------

                eligible_donor_units = (
                    donor_surplus
                    * substitution_redirect_pct
                )


                # -------------------------------------------------------------
                # Conversion rule
                #
                # conversion_ratio =
                # substitute units required per source-SKU equivalent unit.
                #
                # Example:
                # 0.90 donor units / source equivalent
                # -> 900 donor units support 1,000 source-equivalent units.
                # -------------------------------------------------------------

                max_source_recovery = (
                    eligible_donor_units
                    / conversion_ratio
                )


                # -------------------------------------------------------------
                # Do not recover more than the recipient actually needs
                # -------------------------------------------------------------

                substitution_recovered = min(
                    max_source_recovery,
                    recipient_shortage
                )


                # Donor units actually consumed
                donor_units_redirected = (
                    substitution_recovered
                    * conversion_ratio
                )


                # -------------------------------------------------------------
                # Execute substitution transfer
                # -------------------------------------------------------------

                scenario_supply.loc[
                    donor_mask,
                    "total_available_supply"
                ] -= donor_units_redirected

                scenario_supply.loc[
                    recipient_mask,
                    "total_available_supply"
                ] += substitution_recovered


    # -------------------------------------------------------------------------
    # 5. REBUILD FINAL SKU DEMAND / SHORTAGE POSITION
    # -------------------------------------------------------------------------

    drop_columns = [
        "demand_units",
        "shortage_units",
        "shortage_pct",
        "excess_supply_units",
        "supply_status"
    ]

    scenario_supply = scenario_supply.drop(
        columns=[
            column
            for column in drop_columns
            if column in scenario_supply.columns
        ],
        errors="ignore"
    )

    scenario_supply = scenario_supply.merge(
        scenario_demand,
        on="sku_id",
        how="left"
    )

    scenario_supply["demand_units"] = (
        scenario_supply["demand_units"]
        .fillna(0)
    )

    scenario_supply["shortage_units"] = (
        scenario_supply["demand_units"]
        - scenario_supply["total_available_supply"]
    ).clip(lower=0)

    scenario_supply["shortage_pct"] = np.where(
        scenario_supply["demand_units"] > 0,
        scenario_supply["shortage_units"]
        / scenario_supply["demand_units"],
        0
    )

    scenario_supply["excess_supply_units"] = (
        scenario_supply["total_available_supply"]
        - scenario_supply["demand_units"]
    ).clip(lower=0)

    scenario_supply["supply_status"] = np.select(
        [
            scenario_supply["shortage_pct"] >= 0.25,
            scenario_supply["shortage_pct"] > 0
        ],
        [
            "Critical Shortage",
            "Constrained"
        ],
        default="Available"
    )


    # -------------------------------------------------------------------------
    # 6. BUSINESS VALIDATION
    # -------------------------------------------------------------------------

    if substitution_source_sku is not None:

        # Substitution should never create negative supply
        if (
            scenario_supply["total_available_supply"] < -1e-6
        ).any():
            raise ValueError(
                "Substitution created negative SKU supply."
            )


    return (
        scenario_orders,
        scenario_supply,
        float(substitution_recovered)
    )


# =============================================================================
# BALANCED SCENARIO SOLVER
# =============================================================================

def solve_balanced_scenario(
    scenario_orders,
    scenario_supply,
    strategic_target,
    contract_target
):
    """
    Solve the Balanced Allocation policy for one operating scenario.

    The model balances:
    - contribution-margin protection,
    - customer-service fairness,
    - strategic / contractual service commitments.
    """

    model = pulp.LpProblem(
        "Scenario_Balanced_Allocation",
        pulp.LpMaximize
    )


    # -------------------------------------------------------------------------
    # ORDER-LEVEL MODEL INPUTS
    # -------------------------------------------------------------------------

    order_ids = (
        scenario_orders["order_id"]
        .tolist()
    )

    quantities = (
        scenario_orders
        .set_index("order_id")["order_quantity"]
        .astype(float)
        .to_dict()
    )

    margins = (
        scenario_orders
        .set_index("order_id")[
            "contribution_margin_per_unit"
        ]
        .astype(float)
        .to_dict()
    )

    sku_map = (
        scenario_orders
        .set_index("order_id")["sku_id"]
        .to_dict()
    )

    supply_map = (
        scenario_supply
        .set_index("sku_id")[
            "total_available_supply"
        ]
        .astype(float)
        .to_dict()
    )


    # -------------------------------------------------------------------------
    # DECISION VARIABLES
    # -------------------------------------------------------------------------

    x = {
        order_id: pulp.LpVariable(
            f"x_{order_id}",
            lowBound=0,
            upBound=quantities[order_id],
            cat="Continuous"
        )
        for order_id in order_ids
    }


    # -------------------------------------------------------------------------
    # SKU CAPACITY CONSTRAINTS
    # -------------------------------------------------------------------------

    for sku_id, available_supply in supply_map.items():

        sku_orders = [
            order_id
            for order_id in order_ids
            if sku_map[order_id] == sku_id
        ]

        if sku_orders:

            model += (
                pulp.lpSum(
                    x[order_id]
                    for order_id in sku_orders
                )
                <= available_supply
            )


    # -------------------------------------------------------------------------
    # CUSTOMER FAIRNESS & SERVICE COMMITMENTS
    # -------------------------------------------------------------------------

    customer_groups = {
        customer_id: group.copy()
        for customer_id, group
        in scenario_orders.groupby("customer_id")
    }

    minimum_customer_fill = pulp.LpVariable(
        "minimum_customer_fill",
        lowBound=0,
        upBound=1,
        cat="Continuous"
    )

    service_shortfalls = []


    for customer_id, group in customer_groups.items():

        customer_orders = (
            group["order_id"]
            .tolist()
        )

        customer_demand = float(
            group["order_quantity"].sum()
        )

        if customer_demand <= 0:
            continue

        allocated_customer = pulp.lpSum(
            x[order_id]
            for order_id in customer_orders
        )


        # ---------------------------------------------------------------------
        # FAIRNESS FLOOR
        # ---------------------------------------------------------------------

        model += (
            allocated_customer
            >= minimum_customer_fill
            * customer_demand
        )


        # ---------------------------------------------------------------------
        # CUSTOMER SERVICE TARGET
        # ---------------------------------------------------------------------

        is_strategic = bool(
            group["strategic_flag"].iloc[0]
        )

        has_contract = bool(
            group["has_contract"].iloc[0]
        )

        if is_strategic:

            target = strategic_target
            penalty_multiplier = 2.0

        elif has_contract:

            target = contract_target
            penalty_multiplier = 1.0

        else:

            continue


        # ---------------------------------------------------------------------
        # SOFT SERVICE SHORTFALL
        # ---------------------------------------------------------------------

        shortfall = pulp.LpVariable(
            f"shortfall_{customer_id}",
            lowBound=0,
            cat="Continuous"
        )

        model += (
            allocated_customer
            + shortfall
            >= target * customer_demand
        )

        avg_margin = float(
            group[
                "contribution_margin_per_unit"
            ].mean()
        )

        service_shortfalls.append(
            penalty_multiplier
            * avg_margin
            * shortfall
        )


    # -------------------------------------------------------------------------
    # BALANCED OBJECTIVE
    # -------------------------------------------------------------------------

    protected_margin = pulp.lpSum(
        margins[order_id]
        * x[order_id]
        for order_id in order_ids
    )

    total_margin_opportunity = float(
        scenario_orders[
            "contribution_margin_total"
        ].sum()
    )

    fairness_value = (
        minimum_customer_fill
        * total_margin_opportunity
    )

    service_penalty = (
        pulp.lpSum(service_shortfalls)
        if service_shortfalls
        else 0
    )

    model += (
        CONFIG["balanced_margin_weight"]
        * protected_margin
        +
        CONFIG["balanced_fairness_weight"]
        * fairness_value
        -
        service_penalty
    )


    # -------------------------------------------------------------------------
    # SOLVE
    # -------------------------------------------------------------------------

    solver = pulp.PULP_CBC_CMD(
        msg=False,
        timeLimit=30
    )

    model.solve(solver)

    status = pulp.LpStatus[
        model.status
    ]
    if status != "Optimal":
        raise RuntimeError(f"Scenario solve failed: {status}")


    # -------------------------------------------------------------------------
    # EXTRACT ALLOCATION
    # -------------------------------------------------------------------------

    allocation = pd.Series(
        {
            order_id: max(
                float(
                    x[order_id].value()
                    or 0
                ),
                0
            )
            for order_id in order_ids
        },
        name="allocated_units"
    )


    return allocation, status


# =============================================================================
# RUN ONE SCENARIO
# =============================================================================

def run_scenario(
    scenario_name: str,
    demand_multiplier: float = 1.0,
    supply_multiplier: float = 1.0,
    strategic_target: float = None,
    contract_target: float = None,
    substitution_source_sku: str = None
):
    """
    Compare FIFO and Balanced Allocation under one operating scenario.
    """

    # -------------------------------------------------------------------------
    # SERVICE TARGETS
    # -------------------------------------------------------------------------

    strategic_target = (
        CONFIG["min_fill_rate_strategic"]
        if strategic_target is None
        else strategic_target
    )

    contract_target = (
        CONFIG["min_fill_rate_contract"]
        if contract_target is None
        else contract_target
    )


    # -------------------------------------------------------------------------
    # BUILD SCENARIO
    # -------------------------------------------------------------------------

    (
        scenario_orders,
        scenario_supply,
        recovered_units
    ) = rebuild_scenario(
        demand_multiplier=demand_multiplier,
        supply_multiplier=supply_multiplier,
        substitution_source_sku=substitution_source_sku
    )


    # -------------------------------------------------------------------------
    # FIFO
    # -------------------------------------------------------------------------

    fifo_allocation = allocate_fifo(
        scenario_orders,
        scenario_supply
    )

    fifo_kpis, _ = score_allocation(
        scenario_orders,
        fifo_allocation
    )


    # -------------------------------------------------------------------------
    # BALANCED ALLOCATION
    # -------------------------------------------------------------------------

    balanced_allocation, solver_status = (
        solve_balanced_scenario(
            scenario_orders=scenario_orders,
            scenario_supply=scenario_supply,
            strategic_target=strategic_target,
            contract_target=contract_target
        )
    )

    balanced_kpis, _ = score_allocation(
        scenario_orders,
        balanced_allocation
    )


    # -------------------------------------------------------------------------
    # SUPPLY EXPOSURE
    # -------------------------------------------------------------------------

    total_demand = float(
        scenario_supply[
            "demand_units"
        ].sum()
    )

    total_shortage = float(
        scenario_supply[
            "shortage_units"
        ].sum()
    )

    shortage_rate = (
        total_shortage / total_demand
        if total_demand > 0
        else 0
    )


    # -------------------------------------------------------------------------
    # SCENARIO RESULT
    # -------------------------------------------------------------------------

    return {

        "Scenario":
            scenario_name,

        "Solver Status":
            solver_status,

        "Demand Units":
            total_demand,

        "Uncovered Demand":
            total_shortage,

        "Shortage %":
            shortage_rate,

        "Substitution Recovery Units":
            recovered_units,

        "FIFO Fill Rate":
            fifo_kpis["fill_rate"],

        "Balanced Fill Rate":
            balanced_kpis["fill_rate"],

        "FIFO Strategic Service":
            fifo_kpis[
                "strategic_customer_service_level"
            ],

        "Balanced Strategic Service":
            balanced_kpis[
                "strategic_customer_service_level"
            ],

        "FIFO Margin Protected":
            fifo_kpis[
                "contribution_margin_protected"
            ],

        "Balanced Margin Protected":
            balanced_kpis[
                "contribution_margin_protected"
            ],

        "Incremental Margin Protected":
            (
                balanced_kpis[
                    "contribution_margin_protected"
                ]
                -
                fifo_kpis[
                    "contribution_margin_protected"
                ]
            ),

        "FIFO Revenue at Risk":
            fifo_kpis["revenue_at_risk"],

        "Balanced Revenue at Risk":
            balanced_kpis["revenue_at_risk"],

        "Balanced Margin at Risk":
            balanced_kpis["margin_at_risk"],

        "Balanced Contract Penalty":
            balanced_kpis[
                "contract_penalty_exposure"
            ]
    }


print("✓ Scenario and stress-test engine initialized.")

In [ ]:
# =============================================================================
# RUN SCENARIO & STRESS-TEST PORTFOLIO
# =============================================================================

# -----------------------------------------------------------------------------
# SELECT SUBSTITUTION RECOVERY SKU
# Choose the most supply-constrained SKU that has an approved substitution route.
# -----------------------------------------------------------------------------

eligible_substitution_skus = (
    sku_view
    .merge(
        substitutions[["source_sku"]].drop_duplicates(),
        left_on="sku_id",
        right_on="source_sku",
        how="inner"
    )
)

# Prefer an SKU with an actual shortage
eligible_shortages = eligible_substitution_skus[
    eligible_substitution_skus["shortage_units"] > 0
]

if not eligible_shortages.empty:

    scenario5_sku = (
        eligible_shortages
        .sort_values(
            ["shortage_units", "shortage_pct"],
            ascending=False
        )
        .iloc[0]["sku_id"]
    )

elif not eligible_substitution_skus.empty:

    scenario5_sku = (
        eligible_substitution_skus
        .sort_values(
            "shortage_pct",
            ascending=False
        )
        .iloc[0]["sku_id"]
    )

else:
    scenario5_sku = None


# -----------------------------------------------------------------------------
# RUN FIVE MANAGEMENT SCENARIOS
# -----------------------------------------------------------------------------

scenario_results = [

    run_scenario(
        scenario_name="1. Base Case"
    ),

    run_scenario(
        scenario_name="2. Demand Surge (+15%)",
        demand_multiplier=(
            1 + CONFIG["scenario_demand_surge_pct"]
        )
    ),

    run_scenario(
        scenario_name="3. Deeper Supply Disruption (-20%)",
        supply_multiplier=(
            1 - CONFIG["scenario_supply_drop_pct"]
        )
    ),

    run_scenario(
        scenario_name="4. Strategic Customer Protection (98%)",
        strategic_target=CONFIG[
            "scenario_strategic_floor_boost"
        ]
    ),

    run_scenario(
        scenario_name=(
            f"5. Substitution Recovery "
            f"({scenario5_sku or 'No Eligible SKU'})"
        ),
        substitution_source_sku=scenario5_sku
    )
]


# -----------------------------------------------------------------------------
# CONSOLIDATE RESULTS
# -----------------------------------------------------------------------------

scenario_df = pd.DataFrame(
    scenario_results
)


# -----------------------------------------------------------------------------
# EXECUTIVE SCENARIO VIEW
# -----------------------------------------------------------------------------

scenario_display = scenario_df[
    [
        "Scenario",
        "Solver Status",
        "Demand Units",
        "Uncovered Demand",
        "Shortage %",
        "Substitution Recovery Units",
        "FIFO Fill Rate",
        "Balanced Fill Rate",
        "FIFO Strategic Service",
        "Balanced Strategic Service",
        "FIFO Margin Protected",
        "Balanced Margin Protected",
        "Incremental Margin Protected",
        "FIFO Revenue at Risk",
        "Balanced Revenue at Risk",
        "Balanced Margin at Risk",
        "Balanced Contract Penalty"
    ]
].copy()


# -----------------------------------------------------------------------------
# DISPLAY
# -----------------------------------------------------------------------------

print("SCENARIO & STRESS-TEST RESULTS")
print("-" * 80)

display(
    scenario_display.style.format({

        "Demand Units":
            "{:,.0f}",

        "Uncovered Demand":
            "{:,.0f}",

        "Shortage %":
            "{:.1%}",

        "Substitution Recovery Units":
            "{:,.0f}",

        "FIFO Fill Rate":
            "{:.1%}",

        "Balanced Fill Rate":
            "{:.1%}",

        "FIFO Strategic Service":
            "{:.1%}",

        "Balanced Strategic Service":
            "{:.1%}",

        "FIFO Margin Protected":
            "${:,.0f}",

        "Balanced Margin Protected":
            "${:,.0f}",

        "Incremental Margin Protected":
            "${:+,.0f}",

        "FIFO Revenue at Risk":
            "${:,.0f}",

        "Balanced Revenue at Risk":
            "${:,.0f}",

        "Balanced Margin at Risk":
            "${:,.0f}",

        "Balanced Contract Penalty":
            "${:,.0f}"
    })
)

In [ ]:
# =============================================================================
# SCENARIO PERFORMANCE VISUAL
# FIFO vs. Balanced Allocation
# =============================================================================

fig = go.Figure()

fig.add_trace(
    go.Bar(
        name="FIFO",
        x=scenario_df["Scenario"],
        y=scenario_df["FIFO Margin Protected"],
        text=scenario_df["FIFO Margin Protected"],
        texttemplate="$%{text:,.0f}",
        textposition="outside"
    )
)

fig.add_trace(
    go.Bar(
        name="Balanced Allocation",
        x=scenario_df["Scenario"],
        y=scenario_df["Balanced Margin Protected"],
        text=scenario_df["Balanced Margin Protected"],
        texttemplate="$%{text:,.0f}",
        textposition="outside"
    )
)

fig.update_layout(
    barmode="group",
    height=520,
    title="Contribution Margin Protection Across Operating Scenarios",
    xaxis_title="",
    yaxis_title="Contribution Margin Protected ($)",
    yaxis_tickprefix="$",
    yaxis_tickformat="~s",
    legend_title_text="Allocation Policy",
    margin=dict(t=80, b=140)
)

fig.update_xaxes(
    tickangle=-20
)

fig.show()


# =============================================================================
# SCENARIO MANAGEMENT INSIGHTS
# =============================================================================

valid_scenarios = scenario_df[
    scenario_df["Solver Status"].isin(["Optimal", "Feasible"])
].copy()

if not valid_scenarios.empty:

    # Scenario with greatest incremental margin benefit
    best_value_row = valid_scenarios.loc[
        valid_scenarios["Incremental Margin Protected"].idxmax()
    ]

    # Scenario with highest shortage exposure
    highest_shortage_row = valid_scenarios.loc[
        valid_scenarios["Shortage %"].idxmax()
    ]

    # Strategic protection scenario
    strategic_rows = valid_scenarios[
        valid_scenarios["Scenario"].str.contains(
            "Strategic Customer Protection",
            case=False,
            na=False
        )
    ]

    # Substitution scenario
    substitution_rows = valid_scenarios[
        valid_scenarios["Scenario"].str.contains(
            "Substitution Recovery",
            case=False,
            na=False
        )
    ]

    print("SCENARIO MANAGEMENT INSIGHTS")
    print("-" * 70)

    print(
        f"Largest Margin Advantage : "
        f"{best_value_row['Scenario']}"
    )

    print(
        f"Incremental Margin        : "
        f"{CONFIG['currency']}"
        f"{best_value_row['Incremental Margin Protected']:,.0f}"
    )

    print(
        f"Highest Shortage Exposure : "
        f"{highest_shortage_row['Scenario']}"
    )

    print(
        f"Uncovered Demand          : "
        f"{highest_shortage_row['Uncovered Demand']:,.0f} units "
        f"({highest_shortage_row['Shortage %']:.1%})"
    )


    # -------------------------------------------------------------------------
    # STRATEGIC CUSTOMER PROTECTION INSIGHT
    # -------------------------------------------------------------------------

    if not strategic_rows.empty:

        strategic_row = strategic_rows.iloc[0]

        print("\nSTRATEGIC CUSTOMER PROTECTION")
        print("-" * 70)

        print(
            f"Strategic Service         : "
            f"{strategic_row['Balanced Strategic Service']:.1%}"
        )

        print(
            f"Margin Protected          : "
            f"{CONFIG['currency']}"
            f"{strategic_row['Balanced Margin Protected']:,.0f}"
        )


    # -------------------------------------------------------------------------
    # SUBSTITUTION RECOVERY INSIGHT
    # -------------------------------------------------------------------------

    if not substitution_rows.empty:

        substitution_row = substitution_rows.iloc[0]

        print("\nSUBSTITUTION RECOVERY")
        print("-" * 70)

        print(
            f"Recovered Supply          : "
            f"{substitution_row['Substitution Recovery Units']:,.0f} units"
        )

        print(
            f"Balanced Fill Rate        : "
            f"{substitution_row['Balanced Fill Rate']:.1%}"
        )

        print(
            f"Margin Protected          : "
            f"{CONFIG['currency']}"
            f"{substitution_row['Balanced Margin Protected']:,.0f}"
        )

else:

    print(
        "No scenario returned a usable optimization result."
    )



### Scenario Interpretation

The stress-test results show how allocation performance changes as demand, supply availability, customer-service requirements, and substitution options vary.

The analysis highlights where **Balanced Allocation creates the greatest incremental financial value relative to FIFO**, while also identifying the operating conditions that create the highest shortage exposure.

Strategic-customer and substitution scenarios provide additional visibility into the cost of stronger service protection and the potential value of supply-recovery actions.

**Management Interpretation:** Allocation optimization does not remove the underlying supply constraint. Its value comes from directing scarce supply toward the highest-value combination of financial protection, customer commitments, and service priorities while providing management with a quantified view of the resulting trade-offs.

## Supply Uncertainty & Revenue-at-Risk Simulation

Available production during a constrained period is not fully certain. Yield variability, supplier delays, quality holds, and last-minute production losses can materially change the final supply position.

The simulation evaluates a range of plausible supply outcomes around the base plan using the configured uncertainty assumptions and measures the resulting **Revenue-at-Risk under the Balanced Allocation policy**.

Key outputs include:

- Expected Revenue-at-Risk
- Best- and worst-case exposure ranges
- Probability of more severe shortage outcomes
- Sensitivity of customer service to supply variability
- Distribution of financial exposure across simulated supply conditions

**Management Objective:** Quantify the range of potential financial outcomes so leadership can assess the risk around the base allocation plan rather than relying on a single deterministic supply estimate.

In [ ]:
# =============================================================================
# SUPPLY UNCERTAINTY & REVENUE-AT-RISK SIMULATION
# Monte Carlo Risk Assessment | Balanced Allocation Policy
# =============================================================================

mc_rng = np.random.default_rng(
    CONFIG["random_seed"] + 1
)

n_iterations = CONFIG["monte_carlo_iterations"]

supply_std_pct = CONFIG[
    "monte_carlo_supply_std_pct"
]


# -----------------------------------------------------------------------------
# STEP 1 | BUILD BALANCED-POLICY RESPONSE CURVE
# -----------------------------------------------------------------------------
# Rather than solving 500 optimization models, solve the actual Balanced
# Allocation model across a representative range of supply conditions.
# Monte Carlo outcomes are then interpolated from this response curve.
# -----------------------------------------------------------------------------

response_factors = np.linspace(
    0.70,
    1.30,
    25
)

response_records = []

print("Building Balanced Allocation supply-response curve...")


for factor in response_factors:

    scenario_orders_mc, scenario_supply_mc, _ = rebuild_scenario(
        demand_multiplier=1.0,
        supply_multiplier=float(factor)
    )

    allocation_mc, status_mc = solve_balanced_scenario(
        scenario_orders=scenario_orders_mc,
        scenario_supply=scenario_supply_mc,
        strategic_target=CONFIG["min_fill_rate_strategic"],
        contract_target=CONFIG["min_fill_rate_contract"]
    )

    if status_mc not in ["Optimal", "Feasible"]:
        continue

    kpi_mc, _ = score_allocation(
        scenario_orders_mc,
        allocation_mc
    )

    response_records.append({
        "supply_factor": float(factor),
        "revenue_at_risk": float(
            kpi_mc["revenue_at_risk"]
        ),
        "margin_at_risk": float(
            kpi_mc["margin_at_risk"]
        ),
        "fill_rate": float(
            kpi_mc["fill_rate"]
        ),
        "strategic_service": float(
            kpi_mc[
                "strategic_customer_service_level"
            ]
        )
    })


response_curve = (
    pd.DataFrame(response_records)
    .sort_values("supply_factor")
    .reset_index(drop=True)
)

if len(response_curve) < 2:
    raise RuntimeError(
        "Insufficient optimization results to build "
        "the supply-risk response curve."
    )

print(
    f"✓ Response curve completed using "
    f"{len(response_curve)} optimized supply positions."
)


# -----------------------------------------------------------------------------
# STEP 2 | GENERATE SUPPLY UNCERTAINTY
# -----------------------------------------------------------------------------

supply_draws = mc_rng.normal(
    loc=1.0,
    scale=supply_std_pct,
    size=n_iterations
)

# Prevent unrealistic extreme supply assumptions
supply_draws = np.clip(
    supply_draws,
    response_curve["supply_factor"].min(),
    response_curve["supply_factor"].max()
)


# -----------------------------------------------------------------------------
# STEP 3 | MAP MONTE CARLO DRAWS TO BALANCED-POLICY OUTCOMES
# -----------------------------------------------------------------------------

mc_revenue_at_risk = np.interp(
    supply_draws,
    response_curve["supply_factor"],
    response_curve["revenue_at_risk"]
)

mc_margin_at_risk = np.interp(
    supply_draws,
    response_curve["supply_factor"],
    response_curve["margin_at_risk"]
)

mc_fill_rate = np.interp(
    supply_draws,
    response_curve["supply_factor"],
    response_curve["fill_rate"]
)

mc_strategic_service = np.interp(
    supply_draws,
    response_curve["supply_factor"],
    response_curve["strategic_service"]
)


# -----------------------------------------------------------------------------
# STEP 4 | RISK DISTRIBUTION
# -----------------------------------------------------------------------------

p05_rar, p50_rar, p95_rar = np.percentile(
    mc_revenue_at_risk,
    [5, 50, 95]
)

p05_margin, p50_margin, p95_margin = np.percentile(
    mc_margin_at_risk,
    [5, 50, 95]
)

p05_fill, p50_fill, p95_fill = np.percentile(
    mc_fill_rate,
    [5, 50, 95]
)

expected_rar = mc_revenue_at_risk.mean()
expected_margin_risk = mc_margin_at_risk.mean()

# Management escalation threshold:
# revenue-at-risk 15% above the median modeled exposure
risk_threshold = p50_rar * 1.15

probability_above_threshold = (
    mc_revenue_at_risk
    > risk_threshold
).mean()


# -----------------------------------------------------------------------------
# MANAGEMENT RISK SUMMARY
# -----------------------------------------------------------------------------

print("\nSUPPLY UNCERTAINTY — MANAGEMENT RISK ASSESSMENT")
print("-" * 70)

print(f"Simulation Iterations       : {n_iterations:,}")
print(f"Supply Standard Deviation   : {supply_std_pct:.1%}")

print("\nREVENUE AT RISK")
print("-" * 70)

print(
    f"Expected Exposure           : "
    f"{CONFIG['currency']}{expected_rar:,.0f}"
)

print(
    f"5th Percentile              : "
    f"{CONFIG['currency']}{p05_rar:,.0f}"
)

print(
    f"Median Exposure             : "
    f"{CONFIG['currency']}{p50_rar:,.0f}"
)

print(
    f"95th Percentile             : "
    f"{CONFIG['currency']}{p95_rar:,.0f}"
)

print(
    f"Escalation Threshold        : "
    f"{CONFIG['currency']}{risk_threshold:,.0f}"
)

print(
    f"Probability Above Threshold : "
    f"{probability_above_threshold:.1%}"
)


print("\nMARGIN AT RISK")
print("-" * 70)

print(
    f"Expected Margin Exposure    : "
    f"{CONFIG['currency']}{expected_margin_risk:,.0f}"
)

print(
    f"95th Percentile             : "
    f"{CONFIG['currency']}{p95_margin:,.0f}"
)


print("\nCUSTOMER SERVICE")
print("-" * 70)

print(
    f"Median Fill Rate            : "
    f"{p50_fill:.1%}"
)

print(
    f"Downside Fill Rate (P5)     : "
    f"{p05_fill:.1%}"
)


# -----------------------------------------------------------------------------
# MONTE CARLO RESULTS TABLE
# -----------------------------------------------------------------------------

monte_carlo_results = pd.DataFrame({
    "supply_factor": supply_draws,
    "revenue_at_risk": mc_revenue_at_risk,
    "margin_at_risk": mc_margin_at_risk,
    "fill_rate": mc_fill_rate,
    "strategic_service": mc_strategic_service
})


# -----------------------------------------------------------------------------
# REVENUE-AT-RISK DISTRIBUTION
# -----------------------------------------------------------------------------

fig = px.histogram(
    monte_carlo_results,
    x="revenue_at_risk",
    nbins=40,
    title="Revenue-at-Risk Distribution Under Supply Uncertainty",
    labels={
        "revenue_at_risk": "Revenue at Risk ($)"
    }
)

fig.add_vline(
    x=p50_rar,
    line_dash="dash",
    annotation_text="Median"
)

fig.add_vline(
    x=p95_rar,
    line_dash="dot",
    annotation_text="95th Percentile"
)

fig.update_layout(
    height=430,
    xaxis_tickprefix="$",
    xaxis_tickformat="~s",
    yaxis_title="Simulation Frequency",
    showlegend=False
)

fig.show()

## Sensitivity Analysis — Key Decision Drivers

The allocation strategy is tested across the assumptions most likely to influence management decisions: **available supply** and the **strategic-customer service target**.

The sensitivity analysis evaluates how changes in these inputs affect:

- Contribution margin protected
- Revenue and margin at risk
- Strategic-customer service
- Overall fill rate
- Service shortfall exposure

The objective is to identify the assumptions that have the greatest impact on the recommended allocation outcome and determine where management attention should be concentrated.

**Management Objective:** Understand which planning assumptions materially change the value of the Balanced Allocation policy and establish the thresholds at which a different allocation response may be required.

In [ ]:
# =============================================================================
# SENSITIVITY ANALYSIS
# Available Supply × Strategic Customer Service Target
# =============================================================================

# -----------------------------------------------------------------------------
# TEST RANGE
# -----------------------------------------------------------------------------

supply_factors = [
    0.60,
    0.68,
    0.76,
    0.84,
    0.92
]

strategic_targets = [
    0.80,
    0.85,
    0.90,
    0.95,
    0.99
]


# -----------------------------------------------------------------------------
# RUN SENSITIVITY GRID
# -----------------------------------------------------------------------------

sensitivity_records = []

print("Running sensitivity analysis...")

for strategic_target in strategic_targets:

    for target_supply_factor in supply_factors:

        # Convert desired overall production factor into a multiplier
        # relative to the current base production-supply assumption.
        supply_multiplier = (
            target_supply_factor
            / CONFIG["base_supply_factor"]
        )

        scenario_orders_sens, scenario_supply_sens, _ = rebuild_scenario(
            demand_multiplier=1.0,
            supply_multiplier=supply_multiplier
        )

        allocation_sens, status_sens = solve_balanced_scenario(
            scenario_orders=scenario_orders_sens,
            scenario_supply=scenario_supply_sens,
            strategic_target=strategic_target,
            contract_target=CONFIG["min_fill_rate_contract"]
        )

        if status_sens not in ["Optimal", "Feasible"]:
            continue

        kpi_sens, _ = score_allocation(
            scenario_orders_sens,
            allocation_sens
        )

        sensitivity_records.append({
            "Supply Factor": target_supply_factor,
            "Strategic Service Target": strategic_target,
            "Margin Protected":
                kpi_sens["contribution_margin_protected"],
            "Revenue at Risk":
                kpi_sens["revenue_at_risk"],
            "Margin at Risk":
                kpi_sens["margin_at_risk"],
            "Fill Rate":
                kpi_sens["fill_rate"],
            "Strategic Service":
                kpi_sens[
                    "strategic_customer_service_level"
                ]
        })


sensitivity_df = pd.DataFrame(
    sensitivity_records
)

if sensitivity_df.empty:
    raise RuntimeError(
        "Sensitivity analysis produced no usable optimization results."
    )

print(
    f"✓ Sensitivity analysis completed across "
    f"{len(sensitivity_df)} operating combinations."
)


# -----------------------------------------------------------------------------
# BUILD MARGIN-PROTECTION MATRIX
# -----------------------------------------------------------------------------

margin_matrix = (
    sensitivity_df
    .pivot(
        index="Strategic Service Target",
        columns="Supply Factor",
        values="Margin Protected"
    )
    .sort_index()
    .sort_index(axis=1)
)


# -----------------------------------------------------------------------------
# EXECUTIVE HEATMAP
# -----------------------------------------------------------------------------

fig = go.Figure(
    data=go.Heatmap(
        z=margin_matrix.values,
        x=[
            f"{factor:.0%}"
            for factor in margin_matrix.columns
        ],
        y=[
            f"{target:.0%}"
            for target in margin_matrix.index
        ],
        text=np.round(
            margin_matrix.values,
            0
        ),
        texttemplate="$%{text:,.0f}",
        colorbar_title="Margin $"
    )
)

fig.update_layout(
    title=(
        "Contribution Margin Sensitivity to Supply Availability "
        "and Strategic Service Targets"
    ),
    xaxis_title="Production Supply Factor",
    yaxis_title="Strategic Customer Service Target",
    height=470
)

fig.show()


# -----------------------------------------------------------------------------
# DECISION-DRIVER ASSESSMENT
# -----------------------------------------------------------------------------

margin_min = float(
    sensitivity_df["Margin Protected"].min()
)

margin_max = float(
    sensitivity_df["Margin Protected"].max()
)

margin_range = (
    margin_max - margin_min
)


# Average impact of changing supply while holding strategic target constant
supply_sensitivity = (
    sensitivity_df
    .groupby("Strategic Service Target")["Margin Protected"]
    .agg(lambda x: x.max() - x.min())
    .mean()
)

# Average impact of changing strategic target while holding supply constant
strategic_sensitivity = (
    sensitivity_df
    .groupby("Supply Factor")["Margin Protected"]
    .agg(lambda x: x.max() - x.min())
    .mean()
)

most_sensitive_driver = (
    "Available Supply"
    if supply_sensitivity > strategic_sensitivity
    else "Strategic Customer Service Target"
)


# -----------------------------------------------------------------------------
# MANAGEMENT SUMMARY
# -----------------------------------------------------------------------------

print("\nSENSITIVITY ASSESSMENT")
print("-" * 70)

print(
    f"Lowest Margin Protected    : "
    f"{CONFIG['currency']}{margin_min:,.0f}"
)

print(
    f"Highest Margin Protected   : "
    f"{CONFIG['currency']}{margin_max:,.0f}"
)

print(
    f"Total Margin Range         : "
    f"{CONFIG['currency']}{margin_range:,.0f}"
)

print(
    f"Avg. Supply Sensitivity    : "
    f"{CONFIG['currency']}{supply_sensitivity:,.0f}"
)

print(
    f"Avg. Strategic Sensitivity : "
    f"{CONFIG['currency']}{strategic_sensitivity:,.0f}"
)

print(
    f"Primary Decision Driver    : "
    f"{most_sensitive_driver}"
)

## Executive Control Tower — Constrained Supply Allocation

The Executive Control Tower consolidates the key outputs of the allocation analysis into a single management view for the constrained planning period.

The dashboard brings together:

- **Supply Position** — confirmed demand, available supply, shortage units, and overall shortage exposure
- **Allocation Performance** — fill rate, OTIF, backorders, and perfect-order performance
- **Financial Exposure** — contribution margin protected, revenue at risk, margin at risk, and contract-penalty exposure
- **Customer Protection** — strategic and contracted customer service levels
- **Policy Performance** — FIFO versus Balanced Allocation outcomes
- **Risk Indicators** — scenario, sensitivity, and supply-uncertainty results
- **Management Exceptions** — customers, SKUs, and service commitments requiring intervention

The Control Tower is designed to support a structured S&OP decision by showing not only the expected financial benefit of the proposed allocation policy, but also the customer-service and supply risks that remain after optimization.

**Management Objective:** Provide leadership with a concise decision view of where supply is constrained, how available inventory should be allocated, what financial value is protected, which customer commitments remain exposed, and where management action is required before the constrained period begins.

In [ ]:
# =============================================================================
# EXECUTIVE CONTROL TOWER
# Constrained Supply Allocation | Base Case
# =============================================================================

# -----------------------------------------------------------------------------
# EXECUTIVE RISK INDICATORS
# -----------------------------------------------------------------------------

def count_strategic_customers_at_risk(detail, target=0.90, tolerance=1e-6):
    """Count customers below aggregate service target, allowing solver tolerance."""
    totals = detail.loc[detail["strategic_flag"]].groupby("customer_id")[[
        "order_quantity", "allocated_units"
    ]].sum()
    valid = totals["order_quantity"] > 0
    fill = totals.loc[valid, "allocated_units"] / totals.loc[valid, "order_quantity"]
    return int((fill < target - tolerance).sum())

strategic_at_risk_fifo = count_strategic_customers_at_risk(detail_fifo)
strategic_at_risk_balanced = count_strategic_customers_at_risk(detail_balanced)

base_shortage_rate = (
    sku_view["shortage_units"].sum()
    / sku_view["demand_units"].sum()
    if sku_view["demand_units"].sum() > 0
    else 0
)


# -----------------------------------------------------------------------------
# DASHBOARD LAYOUT
# -----------------------------------------------------------------------------

fig = make_subplots(
    rows=2,
    cols=3,
    specs=[
        [
            {"type": "indicator"},
            {"type": "indicator"},
            {"type": "indicator"}
        ],
        [
            {"type": "bar", "colspan": 2},
            None,
            {"type": "bar"}
        ]
    ],
    subplot_titles=(
        "",
        "",
        "",
        "Contribution Margin Protected by Allocation Policy",
        "",
        "Strategic Accounts Below 90% Service"
    ),
    row_heights=[0.35, 0.65]
)


# -----------------------------------------------------------------------------
# KPI CARD 1 | FILL RATE
# -----------------------------------------------------------------------------

fig.add_trace(
    go.Indicator(
        mode="number+delta",
        value=kpi_balanced["fill_rate"] * 100,
        number={
            "suffix": "%",
            "valueformat": ".1f"
        },
        delta={
            "reference": kpi_fifo["fill_rate"] * 100,
            "relative": False,
            "suffix": " pp vs FIFO",
            "valueformat": ".1f"
        },
        title={
            "text": "Balanced Allocation<br>Overall Fill Rate"
        }
    ),
    row=1,
    col=1
)


# -----------------------------------------------------------------------------
# KPI CARD 2 | CONTRIBUTION MARGIN PROTECTED
# -----------------------------------------------------------------------------

fig.add_trace(
    go.Indicator(
        mode="number+delta",
        value=kpi_balanced[
            "contribution_margin_protected"
        ],
        number={
            "prefix": CONFIG["currency"],
            "valueformat": ",.0f"
        },
        delta={
            "reference": kpi_fifo[
                "contribution_margin_protected"
            ],
            "relative": False,
            "prefix": CONFIG["currency"],
            "valueformat": ",.0f"
        },
        title={
            "text": "Balanced Allocation<br>Margin Protected"
        }
    ),
    row=1,
    col=2
)


# -----------------------------------------------------------------------------
# KPI CARD 3 | REVENUE AT RISK
# -----------------------------------------------------------------------------

fig.add_trace(
    go.Indicator(
        mode="number+delta",
        value=kpi_balanced["revenue_at_risk"],
        number={
            "prefix": CONFIG["currency"],
            "valueformat": ",.0f"
        },
        delta={
            "reference": kpi_fifo["revenue_at_risk"],
            "relative": False,
            "prefix": CONFIG["currency"],
            "valueformat": ",.0f"
        },
        title={
            "text": "Balanced Allocation<br>Revenue at Risk"
        }
    ),
    row=1,
    col=3
)


# -----------------------------------------------------------------------------
# POLICY COMPARISON | MARGIN PROTECTION
# -----------------------------------------------------------------------------

fig.add_trace(
    go.Bar(
        x=policy_results["policy"],
        y=policy_results[
            "contribution_margin_protected"
        ],
        text=policy_results[
            "contribution_margin_protected"
        ],
        texttemplate="$%{text:,.0f}",
        textposition="outside",
        showlegend=False
    ),
    row=2,
    col=1
)


# -----------------------------------------------------------------------------
# STRATEGIC SERVICE RISK | FIFO vs. BALANCED
# -----------------------------------------------------------------------------

fig.add_trace(
    go.Bar(
        x=[
            "FIFO",
            "Balanced Allocation"
        ],
        y=[
            strategic_at_risk_fifo,
            strategic_at_risk_balanced
        ],
        text=[
            strategic_at_risk_fifo,
            strategic_at_risk_balanced
        ],
        textposition="outside",
        showlegend=False
    ),
    row=2,
    col=3
)


# -----------------------------------------------------------------------------
# EXECUTIVE FORMAT
# -----------------------------------------------------------------------------

fig.update_yaxes(
    title_text="Contribution Margin Protected ($)",
    tickprefix="$",
    tickformat="~s",
    row=2,
    col=1
)

fig.update_yaxes(
    title_text="Customer Count",
    row=2,
    col=3
)

fig.update_xaxes(
    tickangle=-20,
    row=2,
    col=1
)

fig.update_layout(
    height=680,
    title_text=(
        f"Executive Control Tower — {CONFIG['period_label']} "
        f"| Modeled Shortage Exposure: {base_shortage_rate:.1%}"
    ),
    margin=dict(
        t=100,
        b=110
    ),
    showlegend=False
)

fig.show()


# =============================================================================
# MANAGEMENT SUMMARY
# =============================================================================

print("EXECUTIVE CONTROL TOWER — MANAGEMENT SUMMARY")
print("-" * 72)

print(
    f"Modeled Supply Shortage       : "
    f"{base_shortage_rate:.1%}"
)

print(
    f"Balanced Overall Fill Rate    : "
    f"{kpi_balanced['fill_rate']:.1%}"
)

print(
    f"Balanced Strategic Service    : "
    f"{kpi_balanced['strategic_customer_service_level']:.1%}"
)

print(
    f"Contribution Margin Protected : "
    f"{CONFIG['currency']}"
    f"{kpi_balanced['contribution_margin_protected']:,.0f}"
)

print(
    f"Revenue at Risk               : "
    f"{CONFIG['currency']}"
    f"{kpi_balanced['revenue_at_risk']:,.0f}"
)

print(
    f"Margin at Risk                : "
    f"{CONFIG['currency']}"
    f"{kpi_balanced['margin_at_risk']:,.0f}"
)

print(
    f"Strategic Accounts <90% — FIFO     : "
    f"{strategic_at_risk_fifo}"
)

print(
    f"Strategic Accounts <90% — Balanced : "
    f"{strategic_at_risk_balanced}"
)

## Management Recommendations & Action Priorities

The optimization output is translated into specific actions for the constrained planning period.

The recommendation engine identifies:

- **Customer Escalations** — priority accounts below service targets
- **Supply Recovery** — SKUs requiring production recovery or substitution
- **Allocation Actions** — orders requiring prioritized or reduced allocation
- **Commercial Actions** — customers requiring delivery or partial-shipment discussions
- **Financial Risk** — material revenue, margin, or contract-penalty exposure

**Management Objective:** Convert model results into a prioritized execution plan for Supply Chain, Sales, and Finance.

In [ ]:
# =============================================================================
# MANAGEMENT RECOMMENDATION ENGINE
# Model-Derived Actions & Escalation Priorities
# =============================================================================

recommendation_rows = []


# -----------------------------------------------------------------------------
# 1. STRATEGIC CUSTOMER PROTECTION
# -----------------------------------------------------------------------------

tier1_mask = allocation_matrix["Customer Tier"].isin([
    "Tier 1 - Strategic Contract",
    "Tier 1 - Strategic"
])

tier1_allocated_units = allocation_matrix.loc[
    tier1_mask,
    "Allocated Units"
].sum()

total_allocated_units = allocation_matrix[
    "Allocated Units"
].sum()

tier1_allocation_share = (
    tier1_allocated_units / total_allocated_units
    if total_allocated_units > 0
    else 0
)

recommendation_rows.append({
    "Priority": "High",
    "Action Area": "Customer Protection",
    "Management Action": (
        f"Protect Tier-1 customer allocations representing "
        f"{tier1_allocation_share:.1%} of total allocated supply."
    ),
    "Business Rationale": (
        "Maintains elevated service protection for strategic accounts "
        "during the constrained planning period."
    )
})


# -----------------------------------------------------------------------------
# 2. SUPPLY RECOVERY / SUBSTITUTION
# -----------------------------------------------------------------------------

substitution_rows = scenario_df[
    scenario_df["Scenario"].str.contains(
        "Substitution Recovery",
        case=False,
        na=False
    )
]

base_rows = scenario_df[
    scenario_df["Scenario"].str.startswith(
        "1. Base Case",
        na=False
    )
]

if (
    scenario5_sku is not None
    and not substitution_rows.empty
    and not base_rows.empty
):

    substitution_result = substitution_rows.iloc[0]
    base_result = base_rows.iloc[0]

    recovered_units = float(
        substitution_result[
            "Substitution Recovery Units"
        ]
    )

    fill_rate_gain = (
        substitution_result["Balanced Fill Rate"]
        - base_result["Balanced Fill Rate"]
    )

    margin_gain = (
        substitution_result["Balanced Margin Protected"]
        - base_result["Balanced Margin Protected"]
    )

    recommendation_rows.append({
        "Priority": "High",
        "Action Area": "Supply Recovery",
        "Management Action": (
            f"Activate the approved substitution route for "
            f"{scenario5_sku}."
        ),
        "Business Rationale": (
            f"Modeled recovery of {recovered_units:,.0f} supply units, "
            f"{fill_rate_gain:+.1%} fill-rate improvement, and "
            f"{CONFIG['currency']}{margin_gain:+,.0f} change in "
            f"protected contribution margin."
        )
    })


# -----------------------------------------------------------------------------
# 3. CUSTOMER SERVICE ESCALATION
# -----------------------------------------------------------------------------

if not service_reductions.empty:

    highest_service_risk = (
        service_reductions
        .sort_values(
            "fill_rate_change_pp",
            ascending=True
        )
        .iloc[0]
    )

    recommendation_rows.append({
        "Priority": "High",
        "Action Area": "Commercial Escalation",
        "Management Action": (
            f"Review allocation for "
            f"{highest_service_risk['customer_name']} "
            f"({highest_service_risk['customer_id']})."
        ),
        "Business Rationale": (
            f"Service changes from "
            f"{highest_service_risk['fifo_fill_rate']:.1%} under FIFO "
            f"to {highest_service_risk['optimized_fill_rate']:.1%} "
            f"under Margin Protection "
            f"({highest_service_risk['fill_rate_change_pp']:+.1f} pp)."
        )
    })


# -----------------------------------------------------------------------------
# 4. STRATEGIC SERVICE TARGET TRADE-OFF
# -----------------------------------------------------------------------------

strategic_scenario_rows = scenario_df[
    scenario_df["Scenario"].str.contains(
        "Strategic Customer Protection",
        case=False,
        na=False
    )
]

if (
    not strategic_scenario_rows.empty
    and not base_rows.empty
):

    strategic_result = strategic_scenario_rows.iloc[0]
    base_result = base_rows.iloc[0]

    margin_tradeoff = (
        strategic_result["Balanced Margin Protected"]
        - base_result["Balanced Margin Protected"]
    )

    strategic_service_gain = (
        strategic_result["Balanced Strategic Service"]
        - base_result["Balanced Strategic Service"]
    )

    recommendation_rows.append({
        "Priority": "Medium",
        "Action Area": "Service Policy",
        "Management Action": (
            f"Evaluate increasing the strategic-customer service target "
            f"to {CONFIG['scenario_strategic_floor_boost']:.0%}."
        ),
        "Business Rationale": (
            f"Modeled strategic-service change of "
            f"{strategic_service_gain:+.1%} with a "
            f"{CONFIG['currency']}{margin_tradeoff:+,.0f} impact on "
            f"protected contribution margin."
        )
    })


# -----------------------------------------------------------------------------
# 5. HIGHEST SKU SHORTAGE
# -----------------------------------------------------------------------------

constrained_sku_view = sku_view[
    sku_view["shortage_units"] > 0
].copy()

if not constrained_sku_view.empty:

    highest_shortage_sku = (
        constrained_sku_view
        .sort_values(
            [
                "shortage_units",
                "shortage_pct"
            ],
            ascending=False
        )
        .iloc[0]
    )

    recommendation_rows.append({
        "Priority": "High",
        "Action Area": "Supply Recovery",
        "Management Action": (
            f"Escalate {highest_shortage_sku['sku_id']} "
            f"({highest_shortage_sku['product_family']}) "
            f"for production, supplier, or expedite recovery."
        ),
        "Business Rationale": (
            f"{highest_shortage_sku['shortage_units']:,.0f} units "
            f"of uncovered demand, equivalent to "
            f"{highest_shortage_sku['shortage_pct']:.1%} "
            f"of confirmed SKU demand."
        )
    })


# -----------------------------------------------------------------------------
# 6. MATERIAL FINANCIAL EXPOSURE
# -----------------------------------------------------------------------------

highest_margin_risk_customer = (
    allocation_matrix
    .sort_values(
        "Margin at Risk",
        ascending=False
    )
    .iloc[0]
)

if highest_margin_risk_customer["Margin at Risk"] > 0:

    recommendation_rows.append({
        "Priority": "High",
        "Action Area": "Financial Risk",
        "Management Action": (
            f"Review commercial recovery options for "
            f"{highest_margin_risk_customer['Customer']}."
        ),
        "Business Rationale": (
            f"Customer carries "
            f"{CONFIG['currency']}"
            f"{highest_margin_risk_customer['Revenue at Risk']:,.0f} "
            f"of revenue exposure and "
            f"{CONFIG['currency']}"
            f"{highest_margin_risk_customer['Margin at Risk']:,.0f} "
            f"of contribution margin exposure."
        )
    })


# =============================================================================
# MANAGEMENT ACTION REGISTER
# =============================================================================

management_recommendations = pd.DataFrame(
    recommendation_rows
)

priority_order = {
    "High": 1,
    "Medium": 2,
    "Low": 3
}

management_recommendations["priority_sort"] = (
    management_recommendations["Priority"]
    .map(priority_order)
)

management_recommendations = (
    management_recommendations
    .sort_values(
        [
            "priority_sort",
            "Action Area"
        ]
    )
    .drop(columns="priority_sort")
    .reset_index(drop=True)
)


print("MANAGEMENT ACTION REGISTER")
print("-" * 80)

display(management_recommendations)

## Executive Decision Matrix

The decision matrix consolidates the model outputs into the key actions requiring S&OP leadership review.

Each decision is assessed across:

- **KPI Impact**
- **Financial Impact**
- **Customer / Operational Risk**
- **Management Priority**
- **Recommended Timing**

**Management Objective:** Provide a clear decision framework for approving allocation, supply-recovery, and customer-service actions during the constrained period.

In [ ]:
# =============================================================================
# EXECUTIVE DECISION MATRIX
# S&OP Leadership Review
# =============================================================================

# -----------------------------------------------------------------------------
# SUPPORTING METRICS
# -----------------------------------------------------------------------------

# Base case
base_result = scenario_df[
    scenario_df["Scenario"].str.startswith(
        "1. Base Case",
        na=False
    )
].iloc[0]


# Substitution scenario
substitution_result = scenario_df[
    scenario_df["Scenario"].str.contains(
        "Substitution Recovery",
        case=False,
        na=False
    )
]

if not substitution_result.empty:

    substitution_result = substitution_result.iloc[0]

    substitution_fill_gain = (
        substitution_result["Balanced Fill Rate"]
        - base_result["Balanced Fill Rate"]
    )

    substitution_margin_gain = (
        substitution_result["Balanced Margin Protected"]
        - base_result["Balanced Margin Protected"]
    )

    substitution_recovery_units = float(
        substitution_result["Substitution Recovery Units"]
    )

else:

    substitution_fill_gain = np.nan
    substitution_margin_gain = np.nan
    substitution_recovery_units = 0


# Strategic-service scenario
strategic_result = scenario_df[
    scenario_df["Scenario"].str.contains(
        "Strategic Customer Protection",
        case=False,
        na=False
    )
]

if not strategic_result.empty:

    strategic_result = strategic_result.iloc[0]

    strategic_service_change = (
        strategic_result["Balanced Strategic Service"]
        - base_result["Balanced Strategic Service"]
    )

    strategic_margin_impact = (
        strategic_result["Balanced Margin Protected"]
        - base_result["Balanced Margin Protected"]
    )

else:

    strategic_service_change = np.nan
    strategic_margin_impact = np.nan


# Highest operational shortage
constrained_skus = sku_view[
    sku_view["shortage_units"] > 0
].copy()

if not constrained_skus.empty:

    highest_shortage_sku = (
        constrained_skus
        .sort_values(
            ["shortage_units", "shortage_pct"],
            ascending=False
        )
        .iloc[0]
    )

else:

    highest_shortage_sku = None


# Balanced vs FIFO financial improvement
balanced_margin_gain = (
    kpi_balanced["contribution_margin_protected"]
    - kpi_fifo["contribution_margin_protected"]
)

balanced_strategic_gain = (
    kpi_balanced["strategic_customer_service_level"]
    - kpi_fifo["strategic_customer_service_level"]
)

balanced_fill_gain = (
    kpi_balanced["fill_rate"]
    - kpi_fifo["fill_rate"]
)


# =============================================================================
# BUILD DECISION MATRIX
# =============================================================================

decision_rows = []


# -----------------------------------------------------------------------------
# DECISION 1 | ALLOCATION POLICY
# -----------------------------------------------------------------------------

decision_rows.append({

    "Decision":
        "Adopt Balanced Allocation for the constrained period",

    "Current State":
        "FIFO allocation",

    "Recommended Action":
        "Execute Balanced Allocation using margin, service commitments, "
        "and customer-priority rules.",

    "KPI Impact":
        (
            f"{balanced_fill_gain * 100:+.1f} pp overall fill rate; "
            f"{balanced_strategic_gain * 100:+.1f} pp strategic service"
        ),

    "Financial Impact":
        (
            f"{CONFIG['currency']}"
            f"{balanced_margin_gain:+,.0f} "
            f"contribution margin protected per period"
        ),

    "Risk / Consideration":
        "Requires cross-functional approval of allocation priorities.",

    "Priority":
        "Immediate"
})


# -----------------------------------------------------------------------------
# DECISION 2 | SUBSTITUTION RECOVERY
# -----------------------------------------------------------------------------

if scenario5_sku is not None:

    decision_rows.append({

        "Decision":
            f"Activate approved substitution for {scenario5_sku}",

        "Current State":
            "Approved route not activated",

        "Recommended Action":
            "Redirect eligible substitute supply within approved "
            "conversion limits.",

        "KPI Impact":
            (
                f"{substitution_recovery_units:,.0f} units recovered; "
                f"{substitution_fill_gain * 100:+.1f} pp fill rate"
            ),

        "Financial Impact":
            (
                f"{CONFIG['currency']}"
                f"{substitution_margin_gain:+,.0f} "
                f"contribution margin impact"
            ),

        "Risk / Consideration":
            "Confirm engineering, quality, and customer approval requirements.",

        "Priority":
            "High"
    })


# -----------------------------------------------------------------------------
# DECISION 3 | STRATEGIC SERVICE POLICY
# -----------------------------------------------------------------------------

decision_rows.append({

    "Decision":
        (
            f"Evaluate {CONFIG['scenario_strategic_floor_boost']:.0%} "
            f"strategic-customer service target"
        ),

    "Current State":
        (
            f"{CONFIG['min_fill_rate_strategic']:.0%} "
            f"strategic service target"
        ),

    "Recommended Action":
        "Apply higher protection selectively where relationship risk "
        "justifies the financial trade-off.",

    "KPI Impact":
        (
            f"{strategic_service_change * 100:+.1f} pp "
            f"strategic service"
            if not np.isnan(strategic_service_change)
            else "Scenario result unavailable"
        ),

    "Financial Impact":
        (
            f"{CONFIG['currency']}"
            f"{strategic_margin_impact:+,.0f} "
            f"contribution margin impact"
            if not np.isnan(strategic_margin_impact)
            else "Scenario result unavailable"
        ),

    "Risk / Consideration":
        "Higher service protection may redirect supply away from "
        "higher-margin non-strategic orders.",

    "Priority":
        "Management Review"
})


# -----------------------------------------------------------------------------
# DECISION 4 | SUPPLY RECOVERY
# -----------------------------------------------------------------------------

if highest_shortage_sku is not None:

    decision_rows.append({

        "Decision":
            f"Escalate recovery for "
            f"{highest_shortage_sku['sku_id']}",

        "Current State":
            (
                f"{highest_shortage_sku['shortage_units']:,.0f} "
                f"units uncovered"
            ),

        "Recommended Action":
            "Prioritize supplier expedite, production recovery, "
            "alternate sourcing, or approved substitution.",

        "KPI Impact":
            (
                f"{highest_shortage_sku['shortage_pct']:.1%} "
                f"of SKU demand currently uncovered"
            ),

        "Financial Impact":
            "Reduces concentrated revenue and margin exposure.",

        "Risk / Consideration":
            "Service exposure remains until additional supply is secured.",

        "Priority":
            "Immediate"
    })


# -----------------------------------------------------------------------------
# DECISION 5 | OPERATING GOVERNANCE
# -----------------------------------------------------------------------------

decision_rows.append({

    "Decision":
        "Establish recurring allocation review",

    "Current State":
        "Point-in-time constrained-supply analysis",

    "Recommended Action":
        "Refresh the allocation model when material demand, supply, "
        "or customer-priority assumptions change.",

    "KPI Impact":
        "Keeps allocation and service decisions aligned with current conditions.",

    "Financial Impact":
        "Reduces exposure to outdated allocation decisions.",

    "Risk / Consideration":
        "Requires consistent ERP inputs and S&OP ownership.",

    "Priority":
        "Ongoing"
})


# =============================================================================
# FINAL EXECUTIVE DECISION MATRIX
# =============================================================================

decision_matrix = pd.DataFrame(
    decision_rows
)

priority_order = {
    "Immediate": 1,
    "High": 2,
    "Management Review": 3,
    "Ongoing": 4
}

decision_matrix["priority_sort"] = (
    decision_matrix["Priority"]
    .map(priority_order)
)

decision_matrix = (
    decision_matrix
    .sort_values("priority_sort")
    .drop(columns="priority_sort")
    .reset_index(drop=True)
)


print("EXECUTIVE DECISION MATRIX")
print("-" * 90)

display(decision_matrix)

## Project Architecture

In [ ]:
# =============================================================================
# PROJECT ARCHITECTURE
# End-to-End Decision Workflow
# =============================================================================

stages = [
    "ERP-Style\nData",
    "Data Quality\n& Validation",
    "Shortage &\nKPI Analysis",
    "Allocation\nPolicies",
    "Optimization\nEngine",
    "Scenario &\nRisk Testing",
    "Financial\nImpact",
    "Management\nDecision"
]

n_stages = len(stages)

fig = go.Figure()


# -----------------------------------------------------------------------------
# PROCESS STAGES
# -----------------------------------------------------------------------------

for i, stage in enumerate(stages):

    fig.add_shape(
        type="rect",
        x0=i - 0.42,
        x1=i + 0.42,
        y0=-0.32,
        y1=0.32,
        line=dict(width=1),
        opacity=0.90
    )

    fig.add_annotation(
        x=i,
        y=0,
        text=f"<b>{stage}</b>",
        showarrow=False,
        font=dict(size=11),
        align="center"
    )


# -----------------------------------------------------------------------------
# FLOW CONNECTORS
# -----------------------------------------------------------------------------

for i in range(n_stages - 1):

    fig.add_annotation(
        x=i + 0.56,
        y=0,
        ax=i + 0.43,
        ay=0,
        xref="x",
        yref="y",
        axref="x",
        ayref="y",
        text="",
        showarrow=True,
        arrowhead=3,
        arrowsize=1.2,
        arrowwidth=1.5
    )


# -----------------------------------------------------------------------------
# LAYOUT
# -----------------------------------------------------------------------------

fig.update_xaxes(
    visible=False,
    range=[-0.6, n_stages - 0.4]
)

fig.update_yaxes(
    visible=False,
    range=[-0.7, 0.7]
)

fig.update_layout(
    height=250,
    title={
        "text": "Constrained Supply Allocation — Decision Architecture",
        "x": 0.5,
        "xanchor": "center"
    },
    margin=dict(
        l=20,
        r=20,
        t=70,
        b=20
    ),
    plot_bgcolor="white"
)

fig.show()

## Limitations

- **Synthetic Data** — Customer, order, supply, and financial data are simulated and do not represent an actual company.
- **Planning Scope** — The model represents a single constrained planning period and does not yet include multi-period or multi-plant optimization.
- **Allocation Constraints** — Split allocations are permitted; operational constraints such as pallet sizes, minimum shipment quantities, and logistics capacity are not modeled.
- **Supply Uncertainty** — Risk simulations use modeled supply variability and an optimized response curve rather than live production forecasts.
- **Business Rules** — Contract penalties, substitution eligibility, and service targets are illustrative assumptions.
- **System Integration** — The solution operates as a standalone decision-support model and is not connected to live ERP, APS, or order-management systems.

## Project Documentation

### Overview

A decision-support solution for allocating constrained manufacturing supply across customer orders when confirmed demand exceeds available capacity.

### Business Problem

When supply cannot satisfy all confirmed demand, allocation decisions must balance financial value, contractual commitments, strategic-customer protection, and operational constraints.

The solution replaces rule-based allocation with a structured, optimization-driven decision process.

### Solution Approach

- ERP-style customer, order, inventory, production, and substitution data
- Automated data-quality and business-rule validation
- SKU-level demand and supply exposure analysis
- FIFO and Proportional baseline allocation policies
- Margin Protection, Strategic Customer Protection, and Balanced Allocation models
- Order-level allocation explainability
- Scenario and supply-disruption stress testing
- Substitution recovery analysis
- Supply uncertainty simulation and sensitivity analysis
- Financial impact and Revenue-at-Risk assessment
- Executive Control Tower and management action recommendations

### Technology

**Python | Pandas | NumPy | PuLP / CBC | Plotly | Google Colab**

### Management KPIs

**Fill Rate | OTIF | Backorder Rate | Revenue-at-Risk | Margin-at-Risk | Contribution Margin Protected | Contract Penalty Exposure | Strategic Customer Service**

### Decision Outputs

The solution provides:

- Order-level recommended allocations
- Customer and SKU service-risk visibility
- Financial comparison of alternative allocation policies
- Supply-recovery and substitution priorities
- Scenario and uncertainty exposure
- Management action priorities
- Executive decision matrix

### Business Value

The model demonstrates how optimization-driven allocation can improve the financial use of constrained supply while maintaining explicit protection for strategic and contracted customers.

All financial and operational results are **modeled using synthetic data** and are intended to demonstrate the decision framework rather than represent actual company performance.

## Executive Project Summary

**Business Challenge**  
A constrained supply position requires management to determine which customer orders should receive available inventory while balancing financial value, contractual commitments, and strategic-customer service.

**Solution**  
An optimization-driven allocation system compares FIFO, Proportional, Margin Protection, Strategic Customer Protection, and Balanced Allocation policies to generate order-level allocation recommendations.

**Decision Intelligence**  
The solution combines allocation optimization, explainability, scenario stress testing, supply uncertainty simulation, sensitivity analysis, and financial risk assessment.

**Management Value**  
The model quantifies the trade-offs between margin protection, Revenue-at-Risk, customer service, and supply constraints while identifying priority customer, SKU, substitution, and recovery actions.

**Recommended Direction**  
Use Balanced Allocation as the primary decision framework during constrained periods, supported by targeted supply recovery, approved substitution routes, and management review of material customer-service exceptions.

> **Note:** All results are modeled using synthetic data and demonstrate the decision framework rather than actual company performance.

## Future Enhancements

- **Rolling-Horizon Planning** — extend the model across multiple planning periods with inventory carryover, backorders, and changing supply conditions.

- **Multi-Plant Optimization** — incorporate plant capacity, inventory positioning, and transportation constraints across the manufacturing network.

- **Interactive Planning Interface** — provide business users with scenario controls for supply assumptions, service targets, and allocation priorities without requiring code changes.

- **ERP / APS Integration** — connect the allocation engine to live order, inventory, production, and customer data.

- **Automated Exception Management** — generate alerts when strategic accounts, contracted customers, or critical SKUs are projected to fall below defined service thresholds.